# OMG_Project — LalimSalita RAG Chatbot Development and Evaluation
**ITC 508 | Final Project**  
**Source:** *Talasalitaan Aralin 1–10.pdf*  
**Stack:** PDF extraction · Recursive chunking · multilingual-e5-small · ChromaDB · Groq API

Run cells in order in **Google Colab** or a local Jupyter environment. This notebook is a reproducible experiment *template*: results are produced only when you execute it. Do not report unexecuted or example results as findings.

**Deliverables supported:** three chunking experiments, processing/indexing logs, retrieval evaluation, chatbot responses, CSV/XLSX exports, and source code. IEEE report, responsibility matrix, and signed declaration must be prepared separately.

## 1 · Install dependencies

In [27]:
%pip -q install pypdf chromadb sentence-transformers langchain-text-splitters groq pandas openpyxl

### Purpose
This section installs the Python libraries required to develop and execute the **LalimSalita Retrieval-Augmented Generation (RAG) system**.

These libraries support document processing, text chunking, embedding generation, vector storage, information retrieval, language model integration, and experimental data export.

### Code Explanation

**1. `%pip` — Package Installation Command**
- Executes Python package installation within the current Google Colab notebook environment.
- Ensures that the required libraries are available for import and use in subsequent cells.

**2. `-q` — Quiet Installation Mode**
- Reduces the amount of installation output displayed.
- Keeps the notebook cleaner and easier to read.

**3. `pypdf` — PDF Document Processing**
- Reads and extracts textual content from the uploaded PDF.
- Provides the vocabulary definitions and reference information used as the knowledge source of the RAG system.

**4. `chromadb` — Vector Database**
- Stores document chunks and their numerical embeddings.
- Supports similarity-based retrieval of relevant information.
- Allows the system to retrieve vocabulary content related to the user's question.

**5. `sentence-transformers` — Embedding Model**
- Converts text into numerical vector representations.
- Supports semantic similarity between questions and document chunks.
- Enables the retrieval system to identify related content based on meaning rather than exact keyword matches.

**6. `langchain-text-splitters` — Text Chunking**
- Divides extracted PDF text into smaller document chunks.
- Supports configurable chunk sizes and overlaps.
- Enables experimentation with different chunking settings to assess their effect on retrieval performance.

**7. `groq` — Large Language Model Integration**
- Provides the Python client for communicating with Groq.
- Sends retrieved document context and user questions to the configured language model.
- Receives the generated Filipino vocabulary explanation.

**8. `pandas` — Data Processing and Organization**
- Organizes question-answer records and retrieval results.
- Creates structured tables for experimental analysis.
- Supports exporting results into CSV and Excel formats.

**9. `openpyxl` — Excel File Support**
- Enables reading and writing Excel (.xlsx) files.
- Supports exporting experimental records for manual
  evaluation and documentation.




## 2A · Import libraries and set reproducibility

In [28]:
# ============================================================
# 2A. IMPORT LIBRARIES AND SET REPRODUCIBILITY
# ============================================================

import os, sys, json, time, hashlib, platform, getpass
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from groq import Groq

SEED = 42
np.random.seed(SEED)
ROOT = Path('OMG_Project_outputs')
ROOT.mkdir(exist_ok=True)
print('Libraries imported successfully.')
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())

Libraries imported successfully.
Python: 3.13.18 | Platform: Linux-6.6.159+-x86_64-with-glibc2.39


### Purpose

This section imports the Python libraries required to execute the **LalimSalita Retrieval-Augmented Generation (RAG) system**.

It initializes the dependencies for document processing, text chunking, embedding generation, vector storage, language model integration, and experimental data management.

Additionally, it establishes a fixed random seed for reproducibility and creates a dedicated directory for storing the system's experimental outputs.

---

### Code Explanation

**1. `import os, sys, json, time, hashlib, platform, getpass`**

Imports Python's built-in utility modules for system operations, data handling, performance measurement, and environment management.

- `os` — Provides operating system functionality, including access to environment variables used for API credentials.
- `sys` — Provides information about the Python interpreter, including its version.
- `json` — Supports reading and writing structured JSON data for configurations and experiment records.
- `time` — Measures execution time, such as retrieval latency and answer generation time.
- `hashlib` — Generates cryptographic hashes that can be used to verify document or configuration consistency.
- `platform` — Retrieves operating system and platform details for environment documentation.
- `getpass` — Allows sensitive inputs, such as API keys, to be entered without displaying them.

**Purpose in the RAG System:**
These modules support experiment tracking, runtime monitoring, configuration management, and secure API initialization.

---

**2. `from pathlib import Path`**

Imports the `Path` class for managing filesystem paths.

**Purpose in the RAG System:**
- Defines the directory for saving experimental outputs.
- Simplifies file and folder management.
- Provides platform-independent path handling.

---

**3. `from datetime import datetime, timezone`**

Imports utilities for working with dates and times.

- `datetime` — Generates timestamps for recorded events.
- `timezone` — Supports timezone-aware timestamps, including Coordinated Universal Time (UTC).

**Purpose in the RAG System:**
Records when questions are processed and experimental results are generated.

---

**4. `import numpy as np`**

Imports NumPy, a library for numerical computing.

**Purpose in the RAG System:**
- Supports numerical operations and array processing.
- Provides random seed initialization.
- Helps establish reproducible numerical behavior
  for operations using NumPy's global random generator.

---

**5. `import pandas as pd`**

Imports Pandas for structured data processing.

**Purpose in the RAG System:**
- Organizes question-answer records.
- Stores retrieval results in tabular form.
- Supports CSV and Excel export.
- Facilitates manual evaluation of chunking configurations.

---

**6. `from pypdf import PdfReader`**

Imports the PDF reading utility from the pypdf library.

**Purpose in the RAG System:**
- Opens the uploaded vocabulary reference PDF.
- Extracts text from document pages.
- Provides the textual knowledge source used for chunking,
  embedding, and retrieval.

---

**7. `from sentence_transformers import SentenceTransformer`**

Imports the SentenceTransformer class.

**Purpose in the RAG System:**
- Loads the selected multilingual embedding model.
- Converts document chunks into numerical embeddings.
- Converts user questions into embeddings.
- Supports semantic similarity retrieval.

**Model Used:**
`intfloat/multilingual-e5-small`

---

**8. `from langchain_text_splitters import
RecursiveCharacterTextSplitter`**

Imports the RecursiveCharacterTextSplitter class.

**Purpose in the RAG System:**
- Divides extracted document text into smaller chunks.
- Supports configurable chunk sizes and overlaps.
- Attempts to preserve text boundaries using a hierarchy of separators.
- Enables comparison of different chunking configurations.

---

**9. `import chromadb`**

Imports the ChromaDB vector database library.

**Purpose in the RAG System:**
- Creates and manages vector collections.
- Stores document embeddings and associated metadata.
- Performs similarity-based retrieval.
- Returns relevant document chunks for user questions.

---

**10. `from groq import Groq`**

Imports the Groq API client.

**Purpose in the RAG System:**
- Initializes communication with the Groq API.
- Sends user questions and retrieved document context to the configured large language model.
- Receives generated responses grounded in the retrieved vocabulary information.

**Language Model Used:**
`openai/gpt-oss-20b`

---

### Reproducibility and Output Initialization

**11. `SEED = 42`**

Defines a fixed random seed value.

**Purpose:**
Provides a consistent seed that can be reused across experimental runs.

The value `42` is an arbitrary constant and does not represent a model parameter or retrieval score.

---

**12. `np.random.seed(SEED)`**

Sets NumPy's global random seed to the value 42.

**Purpose:**
- Makes random operations using NumPy's legacy global random generator reproducible.
- Helps reduce variability caused by random number generation.

**Important:**
This does not guarantee complete reproducibility of the embedding model, vector database, or LLM outputs.

---

**13. `ROOT = Path('OMG_Project_outputs')`**

Defines the output directory for the project.

**Purpose:**
Provides a central location for storing experimental files, retrieval records, and exported results.

The directory is relative to the notebook's current working directory.

---

**14. `ROOT.mkdir(exist_ok=True)`**

Creates the output directory if it does not already exist.

**Purpose:**
- Ensures that the output folder is available.
- Prevents an error if the folder already exists.
- Allows subsequent cells to save results to the directory.

---

### Execution Confirmation

**15. `print('Libraries imported successfully.')`**

Displays a confirmation message after the imports and initialization statements execute successfully.

**Purpose:**
Indicates that this cell completed without raising an exception.

---

**16. `print('Python:', sys.version.split()[0],
'| Platform:', platform.platform())`**

Displays information about the runtime environment.

- `sys.version.split()[0]` — Retrieves the Python version.
- `platform.platform()` — Retrieves platform information.

**Purpose in the RAG System:**
Documents the execution environment for experimental reproducibility and technical reporting.

## 2B · Upload PDF document

In [29]:
# ============================================================
# 2B. UPLOAD PDF DOCUMENT
# ============================================================
from google.colab import files

print("\n")
print("=" * 70)
print("UPLOAD DOCUMENT")
print("=" * 70)

uploaded = files.upload()

if len(uploaded) == 0:
    raise RuntimeError("No file was uploaded.")

# Use the first uploaded PDF
pdf_file = next(iter(uploaded.keys()))

if not pdf_file.lower().endswith(".pdf"):
    raise ValueError("The uploaded file is not a PDF. Please upload a PDF.")

PDF_PATH = Path(pdf_file)
print("\nSelected file:")
print(pdf_file)
print('SHA256:', hashlib.sha256(PDF_PATH.read_bytes()).hexdigest())



UPLOAD DOCUMENT


Saving Talasalitaan Dataset (1).pdf to Talasalitaan Dataset (1) (2).pdf

Selected file:
Talasalitaan Dataset (1) (2).pdf
SHA256: 0a4dce6233ad8ee972fd0c43b726e673b96cdd13587d26924e55533ebfb9ae40


### Purpose

This section allows the user to upload the PDF document that will serve as the primary knowledge source of the **LalimSalita Retrieval-Augmented Generation (RAG) system**.

It uses Google Colab's file upload functionality to receive the document, verifies that a file was uploaded, checks its PDF filename extension, and generates a SHA-256 hash to identify the uploaded file.

The selected document will be used in the succeeding stages of PDF text extraction, document chunking, embedding generation, and vector database construction.

---

### Code Explanation

**1. `from google.colab import files`**

Imports the `files` utility from the Google Colab library.

**Purpose in the RAG System:**
- Enables uploading files from the user's local device.
- Provides an interactive upload interface in Google Colab.
- Allows the system to access the uploaded vocabulary PDF within the current Colab runtime.

---

**2. `print("\n")`**

Displays a blank line in the notebook output.

**Purpose:**
Improves the visual spacing before the document upload section is displayed.

---

**3. `print("=" * 70)`**

Prints 70 equal signs to create a horizontal separator.

**Purpose:**
Organizes the notebook output and visually distinguishes the document upload section from other system processes.

---

**4. `print("UPLOAD DOCUMENT")`**

Displays the heading "UPLOAD DOCUMENT".

**Purpose:**
Identifies the current operation being performed by the notebook.

---

**5. `uploaded = files.upload()`**

Opens Google Colab's interactive file upload interface.

The uploaded files are returned as a dictionary containing filenames as keys and file contents as byte values.

**Purpose in the RAG System:**
- Allows the user to select the vocabulary reference PDF.
- Transfers the selected document into the Colab runtime.
- Makes the uploaded document accessible for processing.

---

**6. `if len(uploaded) == 0:`**

Checks whether the upload dictionary is empty.

**Purpose:**
Determines whether the user successfully uploaded at least one file.

If no file was uploaded, the system proceeds to the error-handling statement.

---

**7. `raise RuntimeError("No file was uploaded.")`**

Raises a runtime error when no file is provided.

**Purpose in the RAG System:**
- Prevents subsequent processing without a document.
- Stops execution of the current cell.
- Informs the user that a PDF must be uploaded.

---

**8. `pdf_file = next(iter(uploaded.keys()))`**

Retrieves the filename of the first uploaded file.
- `uploaded.keys()` — Retrieves the uploaded filenames.
- `iter()` — Creates an iterator over those filenames.
- `next()` — Returns the first filename.
- `pdf_file` — Stores the selected filename.

**Purpose in the RAG System:**
Selects one document to serve as the knowledge source for the current experiment.

---

**9. `if not pdf_file.lower().endswith(".pdf"):`**

Checks whether the selected filename ends with `.pdf`.

- `pdf_file.lower()` — Converts the filename to lowercase.
- `.endswith(".pdf")` — Checks its PDF extension.
- `not` — Identifies filenames without that extension.

**Purpose in the RAG System:**
Prevents files with non-PDF filename extensions from proceeding to the PDF processing stage.

---

**10. `raise ValueError("The uploaded file is not a PDF. Please upload a PDF.")`**

Raises a value error when the selected filename does not have a PDF extension.

**Purpose:**
- Stops execution when an unsupported filename is selected.
- Informs the user about the expected file type.
- Reduces the possibility of passing an incompatible file to the PDF extraction stage.

---

**11. `PDF_PATH = Path(pdf_file)`**

Converts the selected filename into a `Path` object.

**Purpose in the RAG System:**
- Provides a reusable reference to the uploaded PDF.
- Allows subsequent cells to locate and open the document.
- Supports filesystem operations through Python's `pathlib` library.

**Dependency:**
The `Path` class was imported in Section 2A.

---

**12. `print("\nSelected file:")`**

Displays a label identifying the selected document.

**Purpose:**
Makes the notebook output easier to interpret.

---

**13. `print(pdf_file)`**

Displays the filename of the selected PDF.

**Purpose in the RAG System:**
- Confirms which document was selected.
- Helps the researcher identify the current data source.
- Supports experimental documentation.

---

**14. `print('SHA256:',
hashlib.sha256(PDF_PATH.read_bytes()).hexdigest())`**

Generates and displays a SHA-256 cryptographic hash
of the selected PDF document.

**Code Breakdown:**
- `PDF_PATH.read_bytes()` — Reads the file as bytes.
- `hashlib.sha256()` — Computes the SHA-256 hash.
- `.hexdigest()` — Converts the hash into a hexadecimal string.
- `print()` — Displays the resulting hash.

**Purpose in the RAG System:**
- Creates a digital fingerprint of the uploaded PDF.
- Helps verify that the same document is used across different experimental runs.
- Supports dataset consistency and reproducibility.

## 3 · PDF extraction, vocabulary formatting, and provenance


In [30]:
# ============================================================
# 3. EXTRACT AND FORMAT VOCABULARY TEXT FROM PDF
# ============================================================

import re

def format_vocabulary_text(raw_text):
    """Normalize PDF whitespace and restore vocabulary-entry/field line breaks.

    Operates independently per PDF page to preserve source-page metadata.
    Does not rewrite the vocabulary words or their definitions.
    """
    text = re.sub(r"\s+", " ", raw_text or "").strip()
    text = re.sub(r"\bARALIN\s+(\d+)\b", r"\n\nARALIN \1\n", text, flags=re.IGNORECASE)
    text = re.sub(r"\bVOCABULARY\s+ENTRY\s+(\d+)\b", r"\n\nVOCABULARY ENTRY \1\n", text, flags=re.IGNORECASE)
    fields = ("WORD", "PART_OF_SPEECH", "DEFINITION", "CONTEXTUAL_MEANING", "STANZA", "SOURCE")
    for field in fields:
        text = re.sub(rf"\s*\b({field})\s*:\s*", r"\n\1: ", text, flags=re.IGNORECASE)
    text = "\n".join(line.strip() for line in text.splitlines())
    return re.sub(r"\n{3,}", "\n\n", text).strip()

t0 = time.perf_counter()
reader = PdfReader(str(PDF_PATH))
pages = []
for page_number, pdf_page in enumerate(reader.pages, start=1):
    raw = pdf_page.extract_text() or ""
    cleaned = format_vocabulary_text(raw)
    if cleaned:
        pages.append({'page': page_number, 'text': cleaned})
if not pages:
    raise ValueError('No extractable text found. Check whether the PDF is scanned.')
extract_seconds = time.perf_counter() - t0

print("\n" + "=" * 70)
print("PDF EXTRACTION AND VOCABULARY FORMATTING")
print("=" * 70)
print(f"Total PDF pages       : {len(reader.pages)}")
print(f"Pages containing text : {len(pages)}")
print(f"Cleaned characters    : {sum(len(p['text']) for p in pages):,}")
print(f"Extraction seconds    : {extract_seconds:.2f}")
print("\n" + "-" * 70)
print("FORMATTED VOCABULARY PREVIEW (PAGE 1)")
print("-" * 70)
print(pages[0]['text'][:1800])
print("\nNote: The preview may end mid-entry; full text is preserved in pages[].")


PDF EXTRACTION AND VOCABULARY FORMATTING
Total PDF pages       : 99
Pages containing text : 99
Cleaned characters    : 90,347
Extraction seconds    : 7.06

----------------------------------------------------------------------
FORMATTED VOCABULARY PREVIEW (PAGE 1)
----------------------------------------------------------------------
ARALIN 1

VOCABULARY ENTRY 001
WORD: namamanglaw
PART_OF_SPEECH: pang-uri
DEFINITION: damdamin ng isang nangungulila; kalagayang malungkot, lalo na kung gabing madilim at sa ilang na pook
CONTEXTUAL_MEANING: malungkot at mapanglaw ang kapaligiran
STANZA: FL-001; KS-05, KS-20
SOURCE: KWF Diksiyonaryo ng Wikang Filipino

VOCABULARY ENTRY 002
WORD: dawag
PART_OF_SPEECH: pangngalan
DEFINITION: alinman sa maraming species ng halamang-baging na matinik at malakahoy
CONTEXTUAL_MEANING: mga matitinik at magkakasalabid na halamang-baging sa kagubatan
STANZA: FL-001, FL-127
SOURCE: KWF Diksiyonaryo ng Wikang Filipino

VOCABULARY ENTRY 003
WORD: masukal
PART_OF_SPEE

### Purpose

This section extracts and formats the vocabulary content
from the uploaded PDF document to prepare it for the
Retrieval-Augmented Generation (RAG) pipeline.

It uses `PdfReader` to extract text from each PDF page
and regular expressions to normalize whitespace and
restore the line structure of vocabulary entries.

The formatting process identifies lesson headings,
vocabulary entry headings, and predefined vocabulary
fields while preserving their associated PDF page numbers.

The resulting structured text serves as the input for
document chunking, embedding generation, and vector
database storage.

---

### Code Explanation

**1. `import re`**

Imports Python's built-in regular expression module.

**Purpose in the RAG System:**
- Identifies text patterns within extracted PDF content.
- Normalizes inconsistent whitespace.
- Restores line breaks around vocabulary headings
  and predefined vocabulary fields.

---

**2. `def format_vocabulary_text(raw_text):`**

Defines a reusable function named
`format_vocabulary_text`.

The function receives raw text extracted from a PDF
page and returns a formatted text string.

**Purpose in the RAG System:**
Prepares vocabulary text for the subsequent chunking
process by improving its structural organization.

---

**3. Function Docstring**

`"""Normalize PDF whitespace and restore
vocabulary-entry/field line breaks."""`

Documents the purpose and intended behavior
of the formatting function.

**Purpose:**
- Explains that the function processes each PDF page
  independently.
- Identifies the preservation of source-page metadata
  as a design objective.
- Clarifies that the function is intended to reformat
  text rather than rewrite vocabulary definitions.

---

**4. `text = re.sub(r"\s+", " ", raw_text or "").strip()`**

Normalizes whitespace in the extracted text.

- `raw_text or ""` — Uses an empty string when
  the input is missing or empty.
- `r"\s+"` — Matches one or more whitespace characters,
  including spaces, tabs, and line breaks.
- `" "` — Replaces each whitespace sequence with
  a single space.
- `.strip()` — Removes leading and trailing whitespace.

**Purpose in the RAG System:**
Creates consistent text spacing before reconstructing
the vocabulary structure.

---

**5. Lesson Heading Formatting**

`re.sub(r"\bARALIN\s+(\d+)\b",
r"\n\nARALIN \1\n", text, flags=re.IGNORECASE)`

Identifies lesson headings such as `ARALIN 1`
and `ARALIN 2`.

- `\bARALIN` — Matches the word ARALIN.
- `\s+` — Matches whitespace after ARALIN.
- `(\d+)` — Captures the lesson number.
- `\1` — Reuses the captured lesson number.
- `\n\n` — Inserts two line breaks before the heading.
- `\n` — Inserts a line break after the heading.
- `re.IGNORECASE` — Allows case-insensitive matching.

**Purpose in the RAG System:**
Separates lessons into recognizable sections
within the extracted text.

---

**6. Vocabulary Entry Heading Formatting**

`re.sub(r"\bVOCABULARY\s+ENTRY\s+(\d+)\b",
r"\n\nVOCABULARY ENTRY \1\n", text,
flags=re.IGNORECASE)`

Identifies vocabulary entry headings and places
them on separate lines.

**Purpose in the RAG System:**
- Distinguishes individual vocabulary entries.
- Improves the readability of extracted content.
- Provides potential text boundaries for
  recursive document chunking.

**Example:**
`VOCABULARY ENTRY 1`

---

**7. `fields = (...)`**

Defines the vocabulary field labels recognized
by the formatting function.

The fields are:

- `WORD` — Vocabulary word.
- `PART_OF_SPEECH` — Grammatical classification.
- `DEFINITION` — Meaning or definition.
- `CONTEXTUAL_MEANING` — Meaning within a specific context.
- `STANZA` — Associated stanza information.
- `SOURCE` — Reference or source information.

**Purpose in the RAG System:**
Identifies the labels used to organize vocabulary
information in the uploaded PDF.

---

**8. `for field in fields:`**

Creates a loop that processes each predefined
vocabulary field.

**Purpose:**
Applies the same formatting rule to all six
recognized field labels.

---

**9. Vocabulary Field Formatting**

`text = re.sub(rf"\s*\b({field})\s*:\s*",
r"\n\1: ", text, flags=re.IGNORECASE)`

Finds each recognized field label followed by a colon
and places it on a new line.

**Code Breakdown:**
- `rf"..."` — Creates a raw formatted string
  containing the current field name.
- `\s*` — Matches optional whitespace.
- `\b` — Identifies a word boundary.
- `({field})` — Captures the matched field label.
- `:` — Matches the colon after the label.
- `\1` — Restores the matched field label.
- `\n` — Inserts a line break before the field.
- `re.IGNORECASE` — Matches labels regardless of case.

**Purpose in the RAG System:**
Organizes vocabulary information into clearly
separated fields.

---

**10. `text = "\n".join(line.strip()
for line in text.splitlines())`**

Removes unnecessary spaces around individual lines
while preserving their line separation.

**Code Breakdown:**
- `.splitlines()` — Divides the text into lines.
- `line.strip()` — Removes surrounding whitespace
  from each line.
- `"\n".join()` — Reassembles the lines using
  newline characters.

**Purpose:**
Produces cleaner and more consistent formatting
for vocabulary entries.

---

**11. `return re.sub(r"\n{3,}", "\n\n", text).strip()`**

Finalizes the formatted text.

**Code Breakdown:**
- `\n{3,}` — Matches three or more consecutive line breaks.
- `\n\n` — Replaces them with two line breaks.
- `.strip()` — Removes surrounding whitespace.
- `return` — Sends the formatted text back
  to the calling process.

**Purpose in the RAG System:**
Prevents excessive blank lines while retaining
the reconstructed vocabulary structure.

---

### PDF Extraction and Processing

**12. `t0 = time.perf_counter()`**

Records a high-resolution starting time.

**Purpose:**
Measures how long the PDF extraction and
formatting process takes.

---

**13. `reader = PdfReader(str(PDF_PATH))`**

Initializes the PDF reader using the uploaded document.

**Code Breakdown:**
- `PDF_PATH` — Path established in Section 2B.
- `str(PDF_PATH)` — Converts the path into a string.
- `PdfReader()` — Opens the PDF for processing.

**Purpose in the RAG System:**
Provides access to the document pages and their
extractable text content.

---

**14. `pages = []`**

Creates an empty list for storing processed pages.

**Purpose:**
Maintains a structured collection of page numbers
and their corresponding formatted text.

---

**15. `for page_number, pdf_page in
enumerate(reader.pages, start=1):`**

Iterates through every page in the PDF document.

**Code Breakdown:**
- `reader.pages` — Accesses the PDF pages.
- `enumerate()` — Provides a page number and page object.
- `start=1` — Begins numbering at page 1.

**Purpose in the RAG System:**
Processes the document page by page while maintaining
the physical PDF page position.

---

**16. `raw = pdf_page.extract_text() or ""`**

Extracts text from the current PDF page.

**Purpose:**
Retrieves the textual content needed for vocabulary
formatting and later RAG processing.

If no text is returned, an empty string is used.

---

**17. `cleaned = format_vocabulary_text(raw)`**

Passes the extracted text to the formatting function.

**Purpose in the RAG System:**
Normalizes whitespace and reconstructs recognized
vocabulary headings and field boundaries.

---

**18. `if cleaned:`**

Checks whether the formatted page contains text.

**Purpose:**
Prevents empty pages from being added to the
processed document collection.

---

**19. `pages.append({'page': page_number,
'text': cleaned})`**

Stores the page number and formatted text
as a dictionary inside the `pages` list.

**Stored Structure:**

`{'page': 1, 'text': 'Formatted vocabulary content'}`

**Purpose in the RAG System:**
- Preserves page-level source information.
- Provides structured input for document chunking.
- Supports source-page identification during retrieval.
- Enables page references in generated answers.

---

**20. `if not pages:`**

Checks whether any pages containing extractable
text were successfully processed.

**Purpose:**
Identifies documents where no usable text
was extracted.

---

**21. `raise ValueError(...)`**

Raises an error if the processed page list is empty.

**Purpose in the RAG System:**
Prevents the chunking and embedding stages
from processing an empty document collection.

The error suggests checking whether the PDF
contains scanned pages.

---

**22. `extract_seconds = time.perf_counter() - t0`**

Calculates the elapsed extraction and formatting time.

**Purpose:**
Records preprocessing execution time for
performance documentation.

---

### Extraction Summary and Output Preview

**23. `print("\n" + "=" * 70)`**

Displays spacing and a horizontal separator.

**Purpose:**
Improves the presentation of notebook results.

---

**24. `print("PDF EXTRACTION AND VOCABULARY FORMATTING")`**

Displays the processing section title.

**Purpose:**
Identifies the completed operation.

---

**25. `print(f"Total PDF pages : {len(reader.pages)}")`**

Displays the total number of pages in the PDF.

**Purpose:**
Documents the size of the source document
in terms of physical PDF pages.

---

**26. `print(f"Pages containing text : {len(pages)}")`**

Displays the number of pages containing
nonempty formatted text.

**Purpose:**
Indicates how many pages were retained
for subsequent processing.

---

**27. `print(f"Cleaned characters :
{sum(len(p['text']) for p in pages):,}")`**

Calculates the total number of characters
across all formatted pages.

- `p['text']` — Retrieves the text of each page.
- `len()` — Counts its characters.
- `sum()` — Adds the character counts.
- `:,` — Formats the result with thousands separators.

**Purpose in the RAG System:**
Measures the amount of formatted text available
for document chunking.

---

**28. `print(f"Extraction seconds :
{extract_seconds:.2f}")`**

Displays the elapsed preprocessing time
with two decimal places.

**Purpose:**
Provides a basic performance measurement
for the extraction and formatting stage.

---

**29. `print("FORMATTED VOCABULARY PREVIEW (PAGE 1)")`**

Displays the heading for the text preview.

**Purpose:**
Allows the researcher to inspect the formatting
of the first retained text-containing page.

---

**30. `print(pages[0]['text'][:1800])`**

Displays the first 1,800 characters from
the first retained page.

**Code Breakdown:**
- `pages[0]` — Accesses the first stored page.
- `['text']` — Retrieves its formatted text.
- `[:1800]` — Selects the first 1,800 characters.

**Purpose in the RAG System:**
Allows visual inspection of vocabulary headings,
field labels, definitions, and text formatting.

**Important:**
The preview may stop in the middle of an entry.
The complete formatted text remains in `pages`.

---

**31. Final Preview Note**

`print("\nNote: The preview may end mid-entry;
full text is preserved in pages[].")`

Displays a clarification about the preview.

**Purpose:**
Confirms that the 1,800-character limit applies
only to the displayed preview, not the stored text.



## 4 · Chunking Configuration


In [31]:
# ============================================================
# 4. ONE CHUNKING CONFIGURATION PER RUN
# ============================================================

# Change only this dictionary for your next experiment.
ACTIVE_CONFIG = {'id': 'CFG3', 'chunk_size': 500, 'chunk_overlap': 50}
TOP_K = 3
EMBED_MODEL = 'intfloat/multilingual-e5-small'
GROQ_MODEL = 'openai/gpt-oss-20b'
TEMPERATURE = 0.2
DEFAULT_CONFIG = ACTIVE_CONFIG['id']
if not (ACTIVE_CONFIG['chunk_size'] > ACTIVE_CONFIG['chunk_overlap'] >= 0):
    raise ValueError(f"Invalid chunking configuration: {ACTIVE_CONFIG}")
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9._-]{1,510}[A-Za-z0-9]', DEFAULT_CONFIG):
    raise ValueError(f"Invalid ChromaDB collection name: {DEFAULT_CONFIG}")
print("\n" + "=" * 70)
print("ACTIVE CHUNKING CONFIGURATION")
print("=" * 70)
print(pd.DataFrame([ACTIVE_CONFIG]).to_string(index=False))
print('Embedding model :', EMBED_MODEL)
print('Top-K retrieval :', TOP_K)


ACTIVE CHUNKING CONFIGURATION
  id  chunk_size  chunk_overlap
CFG3         500             50
Embedding model : intfloat/multilingual-e5-small
Top-K retrieval : 3


### Purpose

This section defines the active experimental configuration for the LalimSalita Retrieval-Augmented Generation (RAG) system.

It allows the researcher to select one chunking configuration per experimental run by modifying the `ACTIVE_CONFIG` dictionary.

The configuration specifies the chunk size, chunk overlap, retrieval limit, embedding model, language model, and generation temperature.

It also validates the selected chunking parameters and ChromaDB collection name before proceeding to the document chunking and retrieval stages.

---

### Code Explanation

**1. `ACTIVE_CONFIG = {'id': 'CFG1', 'chunk_size': 300, 'chunk_overlap': 50}`**

Defines the active chunking configuration using a Python dictionary.

- `ACTIVE_CONFIG` — Stores the configuration selected for the current experiment.
- `'id': 'CFG1'` — Assigns a unique identifier to the configuration.
- `'chunk_size': 300` — Specifies the maximum target chunk length in characters.
- `'chunk_overlap': 50` — Specifies the target number of overlapping characters between adjacent chunks.

**Purpose in the RAG System:**
- Controls how the extracted vocabulary text will be divided into smaller chunks.
- Enables systematic experimentation with different chunk sizes and overlaps.
- Identifies the configuration used in each experiment.

---

**2. `TOP_K = 3`**

Defines the maximum number of document chunks requested during each retrieval operation.

**Purpose in the RAG System:**
- Limits retrieval to the top three matching chunks.
- Controls how much retrieved context is supplied to the language model.
- Establishes a consistent retrieval cutoff across chunking experiments.

---

**3. `EMBED_MODEL = 'intfloat/multilingual-e5-small'`**

Specifies the multilingual embedding model used by the RAG system.

**Purpose in the RAG System:**
- Converts vocabulary chunks into numerical vectors.
- Converts user questions into numerical vectors.
- Supports semantic similarity retrieval.
- Enables matching based on semantic relationships.

---

**4. `GROQ_MODEL = 'openai/gpt-oss-20b'`**

Specifies the language model identifier used for answer generation through Groq.

**Purpose in the RAG System:**
- Identifies the model that will generate answers.
- Supports Filipino vocabulary question answering.
- Uses retrieved document context as the basis for generating responses.

---

**5. `TEMPERATURE = 0.2`**

Defines the temperature parameter intended for language model generation.

**Purpose in the RAG System:**
- Controls the sampling behavior of the language model when the parameter is supported and passed to the API.
- Uses a relatively low temperature to encourage more consistent responses.
- Reduces, but does not eliminate, generation variability.

---

**6. `DEFAULT_CONFIG = ACTIVE_CONFIG['id']`**

Retrieves the identifier of the active configuration.

**Code Breakdown:**
- `ACTIVE_CONFIG['id']` — Accesses the selected configuration identifier.
- `DEFAULT_CONFIG` — Stores that identifier for use in later processing stages.

**Purpose in the RAG System:**
- Provides a reusable configuration identifier.
- Allows downstream functions to reference the active experiment.
- Supports consistent configuration labeling in retrieval and evaluation records.

---

### Chunking Configuration Validation

**7. `if not (ACTIVE_CONFIG['chunk_size'] > ACTIVE_CONFIG['chunk_overlap'] >= 0):`**

Checks whether the chunking parameters satisfy the required numerical relationship.

The validation requires:

`chunk_size > chunk_overlap >= 0`

**Purpose in the RAG System:**
- Ensures the overlap is not negative.
- Ensures the overlap is smaller than the chunk size.
- Prevents invalid chunking parameter combinations.

---

**8. `raise ValueError(f"Invalid chunking configuration: {ACTIVE_CONFIG}")`**

Raises an error if the chunking validation fails.

**Purpose:**
- Stops the current cell when parameters are invalid.
- Displays the configuration responsible for the error.
- Prevents downstream processing with invalid settings.

---

**9. `if not re.fullmatch( r'[A-Za-z0-9][A-Za-z0-9._-]{1,510}[A-Za-z0-9]', DEFAULT_CONFIG):`**

Validates the configuration identifier against a regular expression used for ChromaDB naming.

**Code Breakdown:**
- `re.fullmatch()` — Checks whether the entire identifier matches the pattern.
- `[A-Za-z0-9]` — Requires an alphanumeric first character.
- `[A-Za-z0-9._-]{1,510}` — Allows letters, numbers, periods, underscores, and hyphens in the middle.
- `[A-Za-z0-9]` — Requires an alphanumeric final character.

**Purpose in the RAG System:**
- Checks the collection identifier's allowed format.
- Reduces the possibility of invalid collection names.
- Supports consistent naming of experimental collections.

---

**10. `raise ValueError(f"Invalid ChromaDB collection name: {DEFAULT_CONFIG}")`**

Raises an error if the configuration identifier does not match the required pattern.

**Purpose:**
Prevents the notebook from proceeding with a rejected collection identifier.

---

### Display Active Configuration

**11. `print("\n" + "=" * 70)`**

Displays a blank line and a horizontal separator.

**Purpose:**
Improves the organization and readability of the Google Colab output.

---

**12. `print("ACTIVE CHUNKING CONFIGURATION")`**

Displays the configuration section heading.

**Purpose:**
Identifies the experiment settings currently selected by the researcher.

---

**13. `print("=" * 70)`**

Displays another horizontal separator.

**Purpose:**
Creates a consistent section header.

---

**14. `print(pd.DataFrame([ACTIVE_CONFIG]).to_string(index=False))`**

Converts the active configuration dictionary into a Pandas DataFrame and displays it as a table.

**Code Breakdown:**
- `[ACTIVE_CONFIG]` — Wraps the dictionary in a list.
- `pd.DataFrame()` — Creates a tabular representation.
- `.to_string()` — Converts the table into printable text.
- `index=False` — Hides the row index.

**Purpose in the RAG System:**
Provides a readable summary of the selected chunking configuration.

---

**15. `print('Embedding model :', EMBED_MODEL)`**

Displays the selected embedding model identifier.

**Purpose:**
Documents the embedding model assigned to the current experimental run.

---

**16. `print('Top-K retrieval :', TOP_K)`**

Displays the retrieval cutoff.

**Purpose:**
Confirms the number of document chunks requested during retrieval.


## 5 · Load embedding model (shared across experiments)

In [32]:
t0 = time.perf_counter()
encoder = SentenceTransformer(EMBED_MODEL)
embed_load_seconds = time.perf_counter()-t0
print('Loaded:', EMBED_MODEL, '| dimension:', encoder.get_sentence_embedding_dimension(), '| seconds:', round(embed_load_seconds,2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: intfloat/multilingual-e5-small | dimension: 384 | seconds: 5.24


/tmp/ipykernel_15670/818795618.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Loaded:', EMBED_MODEL, '| dimension:', encoder.get_sentence_embedding_dimension(), '| seconds:', round(embed_load_seconds,2))


### Purpose

This section initializes the multilingual embedding model used in the LalimSalita Retrieval-Augmented Generation (RAG) system.

The embedding model converts text into numerical vector representations that capture semantic relationships between vocabulary definitions and user questions.

The selected model is:

`intfloat/multilingual-e5-small`

This section also measures the time required to load the model and identifies the dimensionality of its generated embeddings.

These operations prepare the embedding component for the subsequent vector generation and similarity retrieval stages.

---

### Code Explanation

**1. `t0 = time.perf_counter()`**

Records the starting time before loading the embedding model.

- `time` — Python's built-in time module.
- `perf_counter()` — Returns a high-resolution performance counter.
- `t0` — Stores the starting counter value.

**Purpose in the RAG System:**
- Establishes the beginning of model initialization.
- Enables measurement of model loading time.
- Supports runtime performance documentation.

---

**2. `encoder = SentenceTransformer(EMBED_MODEL)`**

Loads the selected SentenceTransformer embedding model.

**Code Breakdown:**
- `SentenceTransformer()` — Initializes the embedding model using the Sentence Transformers library.
- `EMBED_MODEL` — Contains the selected model identifier defined in Section 4.
- `encoder` — Stores the initialized model object.

**Selected Model:**
`intfloat/multilingual-e5-small`

**Purpose in the RAG System:**
- Prepares the model for generating text embeddings.
- Enables conversion of Filipino vocabulary content into numerical vector representations.
- Supports semantic similarity between user questions and document chunks.

---

**3. `embed_load_seconds = time.perf_counter() - t0`**

Calculates the elapsed time required to initialize the embedding model.

- `time.perf_counter()` — Retrieves the current
  performance counter value.
- `t0` — Contains the previously recorded start time.
- Subtraction — Calculates the elapsed duration.
- `embed_load_seconds` — Stores the result in seconds.

**Purpose in the RAG System:**
- Measures embedding model initialization time.
- Supports monitoring of system setup performance.
- Helps document the computational requirements
  of the embedding component.

---

**4. `print('Loaded:', EMBED_MODEL, ...)`**

Displays the identifier of the successfully initialized embedding model.

**Purpose:**
Confirms which embedding model is being used for the current experiment.

---

**5. `encoder.get_sentence_embedding_dimension()`**

Retrieves the dimensionality of the vectors produced by the embedding model.

**Code Breakdown:**
- `encoder` — The initialized embedding model.
- `.get_sentence_embedding_dimension()` — Returns the model's output embedding dimension.

**Expected Model Dimension:**
384 dimensions for `intfloat/multilingual-e5-small`.

**Purpose in the RAG System:**
- Identifies the size of each generated vector.
- Helps verify embedding compatibility.
- Supports correct vector storage and retrieval within ChromaDB.

---

**6. `round(embed_load_seconds, 2)`**

Rounds the measured model loading time to two decimal places.

**Purpose:**
Improves the readability of the performance measurement displayed in Google Colab.

The original measurement remains available
in the `embed_load_seconds` variable.

---

**7. Complete Print Statement**

`print('Loaded:', EMBED_MODEL,
'| dimension:',
encoder.get_sentence_embedding_dimension(),
'| seconds:', round(embed_load_seconds, 2))`

Displays the embedding model's initialization summary.

**Output Information:**
- Loaded model identifier.
- Embedding vector dimensionality.
- Model initialization time in seconds.

**Purpose in the RAG System:**
Provides confirmation that the embedding model
has been initialized and reports its basic
technical characteristics.

## 6 · Build and inspect the selected ChromaDB collection

In [33]:
# ============================================================
# 6. BUILD AND DISPLAY CHUNKING EXPERIMENTS (CHROMADB)
# ============================================================
client = chromadb.PersistentClient(path=str(ROOT / 'chroma_db'))
collections = {}
chunk_rows, experiment_logs = [], []

for cfg in [ACTIVE_CONFIG]:
    config_id = cfg['id']
    print("\n" + "=" * 70)
    print(f"CONFIGURATION {config_id} | SIZE {cfg['chunk_size']} | OVERLAP {cfg['chunk_overlap']}")
    print("=" * 70)

    t0 = time.perf_counter()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
        separators=['\n\n', '\n', ' ', ''],
    )
    chunks = []
    for p in pages:
        for j, part in enumerate(splitter.split_text(p['text'])):
            chunks.append({'id': f"{config_id}_p{p['page']}_c{j}",
                           'page': p['page'], 'text': part})
    if not chunks:
        raise RuntimeError(f"No chunks generated for {config_id}.")
    split_seconds = time.perf_counter() - t0
    lengths = [len(x['text']) for x in chunks]

    print("\n" + "-" * 70)
    print("CHUNKING RESULTS")
    print("-" * 70)
    print(f"Chunk size             : {cfg['chunk_size']} characters")
    print(f"Chunk overlap          : {cfg['chunk_overlap']} characters")
    print(f"Source pages           : {len(pages)}")
    print(f"Number of chunks       : {len(chunks)}")
    print(f"Average characters     : {np.mean(lengths):.2f}")
    print(f"Shortest chunk         : {min(lengths)} characters")
    print(f"Longest chunk          : {max(lengths)} characters")
    print(f"Chunking seconds       : {split_seconds:.4f}")

    print("\n" + "-" * 70)
    print(f"SAMPLE CHUNKS — {config_id} (FIRST 3)")
    print("-" * 70)
    for chunk in chunks[:3]:
        print(f"\nChunk ID   : {chunk['id']}")
        print(f"Page       : {chunk['page']}")
        print(f"Characters : {len(chunk['text'])}")
        print("-" * 70)
        print(chunk['text'])

    t1 = time.perf_counter()
    vectors = encoder.encode(
        ['passage: ' + x['text'] for x in chunks],
        normalize_embeddings=True, batch_size=32, show_progress_bar=False,
    )
    embedding_seconds = time.perf_counter() - t1

    t2 = time.perf_counter()
    try:
        client.delete_collection(config_id)
    except (ValueError, chromadb.errors.NotFoundError):
        pass
    col = client.create_collection(name=config_id, metadata={'hnsw:space': 'cosine'})
    for start in range(0, len(chunks), 100):
        batch = chunks[start:start + 100]
        col.add(
            ids=[x['id'] for x in batch],
            documents=[x['text'] for x in batch],
            metadatas=[{'page': x['page'], 'config': config_id} for x in batch],
            embeddings=vectors[start:start + 100].tolist(),
        )
    index_seconds = time.perf_counter() - t2
    collections[config_id] = col
    experiment_logs.append({
        **cfg, 'pages': len(pages), 'chunks': len(chunks),
        'mean_chunk_chars': round(float(np.mean(lengths)), 1),
        'min_chunk_chars': min(lengths), 'max_chunk_chars': max(lengths),
        'split_seconds': round(split_seconds, 4),
        'embedding_seconds': round(embedding_seconds, 4),
        'index_seconds': round(index_seconds, 4),
        'vector_dimension': int(vectors.shape[1]),
    })
    chunk_rows.extend({'config': config_id, **x, 'characters': len(x['text'])} for x in chunks)

    print("\n" + "-" * 70)
    print("EMBEDDING AND VECTOR DATABASE RESULTS")
    print("-" * 70)
    print(f"Embedding model        : {EMBED_MODEL}")
    print(f"Vector dimensions      : {vectors.shape[1]}")
    print(f"Indexed Chroma records : {col.count()}")
    print(f"Embedding seconds      : {embedding_seconds:.4f}")
    print(f"Indexing seconds       : {index_seconds:.4f}")

print("\n" + "=" * 70)
print("CURRENT CONFIGURATION SUMMARY")
print("=" * 70)
print(pd.DataFrame(experiment_logs).to_string(index=False))


CONFIGURATION CFG3 | SIZE 500 | OVERLAP 50

----------------------------------------------------------------------
CHUNKING RESULTS
----------------------------------------------------------------------
Chunk size             : 500 characters
Chunk overlap          : 50 characters
Source pages           : 99
Number of chunks       : 274
Average characters     : 329.08
Shortest chunk         : 154 characters
Longest chunk          : 498 characters
Chunking seconds       : 0.0019

----------------------------------------------------------------------
SAMPLE CHUNKS — CFG3 (FIRST 3)
----------------------------------------------------------------------

Chunk ID   : CFG3_p1_c0
Page       : 1
Characters : 319
----------------------------------------------------------------------
ARALIN 1

VOCABULARY ENTRY 001
WORD: namamanglaw
PART_OF_SPEECH: pang-uri
DEFINITION: damdamin ng isang nangungulila; kalagayang malungkot, lalo na kung gabing madilim at sa ilang na pook
CONTEXTUAL_MEANING: malung

### Purpose

This section implements the document chunking, embedding
generation, and vector database construction stages of
the LalimSalita Retrieval-Augmented Generation (RAG) system.

It processes the formatted vocabulary text extracted
from the PDF using the active chunking configuration
defined in Section 4.

The section performs the following operations:

1. Initializes a persistent ChromaDB vector database.
2. Divides the vocabulary text into document chunks.
3. Preserves the source PDF page number of each chunk.
4. Calculates chunking statistics and execution time.
5. Displays sample chunks for manual inspection.
6. Converts chunks into numerical vector embeddings.
7. Creates a ChromaDB collection using cosine distance.
8. Stores embeddings, chunk text, and metadata.
9. Records experimental performance measurements.
10. Displays the active configuration summary.

Only one configuration is processed per run.

---

### Code Explanation

#### A. ChromaDB Initialization

**1. `client = chromadb.PersistentClient(
path=str(ROOT / 'chroma_db'))`**

Initializes a persistent ChromaDB database client.

**Code Breakdown:**
- `chromadb.PersistentClient()` — Creates a ChromaDB client that stores data on the filesystem.
- `ROOT` — Output directory defined in Section 2A.
- `ROOT / 'chroma_db'` — Specifies the database folder.
- `str()` — Converts the path into a string.

**Purpose in the RAG System:**
- Establishes the vector database environment.
- Allows storage of vocabulary embeddings.
- Supports later similarity-based retrieval.
- Maintains the database within the configured output directory.

---

**2. `collections = {}`**

Creates an empty Python dictionary.

**Purpose:**
Stores references to the ChromaDB collections created during the current notebook execution.

**Example:**
`collections['CFG1']`

This allows the retrieval function to access the collection associated with the active chunking configuration.

---

**3. `chunk_rows, experiment_logs = [], []`**

Initializes two empty lists.

**Code Breakdown:**
- `chunk_rows` — Stores individual chunk records.
- `experiment_logs` — Stores experiment-level
  statistics and performance measurements.

**Purpose in the RAG System:**
Organizes the generated data for later analysis and CSV or Excel export.

---

#### B. Active Configuration Processing

**4. `for cfg in [ACTIVE_CONFIG]:`**

Starts a loop using the active configuration defined in Section 4.

**Purpose:**
Processes exactly one selected configuration during the current execution.

---

**5. `config_id = cfg['id']`**

Retrieves the active configuration identifier.

**Purpose:**
Provides a consistent identifier for:
- Generated chunks.
- ChromaDB collection names.
- Experimental records.
- Exported results.

---

**6. Configuration Display Statements**

`print("=" * 70)`

`print(f"CONFIGURATION {config_id} ...")`

Displays the active configuration identifier, chunk size, and overlap.

**Purpose:**
Makes the experiment settings visible before the chunking process begins.

---

#### C. Recursive Text Chunking

**7. `t0 = time.perf_counter()`**

Records the starting time for chunk generation.

**Purpose:**
Enables measurement of the chunking operation's execution time.

---

**8. `splitter = RecursiveCharacterTextSplitter(...)`**

Initializes the recursive character-based text splitting algorithm.

**Parameters:**

`chunk_size=cfg['chunk_size']`

- Defines the maximum target chunk length.
- Uses the value selected in Section 4.
- Measured in characters by default.

`chunk_overlap=cfg['chunk_overlap']`

- Defines the target overlap between neighboring chunks.
- Helps retain context across chunk boundaries.

`separators=['\n\n', '\n', ' ', '']`

Defines the order of preferred text separators:

1. `\n\n` — Paragraph or section boundaries.
2. `\n` — Individual line boundaries.
3. `' '` — Word boundaries.
4. `''` — Character-level splitting.

**Purpose in the RAG System:**
- Divides long vocabulary content into smaller retrievable text units.
- Attempts to preserve natural text boundaries.
- Applies the experimental chunk size and overlap settings.

---

**9. `chunks = []`**

Creates an empty list for generated chunks.

**Purpose:**
Collects all chunks produced from the formatted PDF pages.

---

**10. `for p in pages:`**

Iterates through the formatted page records created in Section 3.

**Purpose:**
Processes each PDF page independently, preserving its original page reference.

---

**11. `for j, part in enumerate( splitter.split_text(p['text'])):`**

Splits the current page into chunks and assigns an index to each generated chunk.

- `p['text']` — Formatted text from the current page.
- `splitter.split_text()` — Performs text chunking.
- `enumerate()` — Assigns an index to each chunk.
- `j` — Chunk index within the page.
- `part` — Text content of the generated chunk.

**Purpose:**
Creates retrievable text segments while tracking their order within each page.

---

**12. `chunks.append({...})`**

Adds a chunk record to the `chunks` list.

Each record contains:

- `id` — Unique chunk identifier.
- `page` — Source PDF page number.
- `text` — Chunk content.

**Example Structure:**

`{'id': 'CFG1_p13_c0',
'page': 13,
'text': 'Vocabulary content...'}`

**Purpose in the RAG System:**
- Preserves the relationship between chunks and their source pages.
- Supports retrieval traceability.
- Allows retrieved chunks to be identified during manual evaluation.

---

**13. `if not chunks:`**

Checks whether the chunking process produced at least one chunk.

**Purpose:**
Prevents empty chunk collections from proceeding to embedding generation.

---

**14. `raise RuntimeError(...)`**

Stops execution when no chunks were generated.

**Purpose:**
Ensures that the embedding and indexing stages receive valid text inputs.

---

**15. `split_seconds = time.perf_counter() - t0`**

Calculates the elapsed chunking time.

**Purpose:**
Measures the time required to divide the formatted PDF text into chunks.

---

**16. `lengths = [len(x['text']) for x in chunks]`**

Creates a list containing the character length of every generated chunk.

**Purpose:**
Provides the data needed to calculate chunk size statistics.

---

#### D. Chunking Statistics

**17. `print("CHUNKING RESULTS")`**

Displays the chunking results heading.

**Purpose:**
Separates the chunking statistics from other notebook outputs.

---

**18. `print(f"Chunk size ...")`**

Displays the configured chunk size.

**Purpose:**
Documents the maximum target chunk length used in the experiment.

---

**19. `print(f"Chunk overlap ...")`**

Displays the configured overlap.

**Purpose:**
Documents the intended context overlap between neighboring chunks.

---

**20. `print(f"Source pages ...")`**

Displays the number of formatted source pages.

**Purpose:**
Identifies how many PDF pages were included in the chunking operation.

---

**21. `print(f"Number of chunks ...")`**

Displays the total number of generated chunks.

**Purpose in the RAG System:**
Measures the number of text segments created under the active configuration.

---

**22. `np.mean(lengths)`**

Calculates the average character length of the generated chunks.

**Purpose:**
Provides a descriptive measure of actual chunk lengths.

---

**23. `min(lengths)`**

Identifies the shortest generated chunk.

**Purpose:**
Helps identify unusually small text segments that may contain limited context.

---

**24. `max(lengths)`**

Identifies the longest generated chunk.

**Purpose:**
Confirms the largest chunk produced by the selected splitting configuration.

---

**25. `split_seconds:.4f`**

Formats the chunking time to four decimal places.

**Purpose:**
Provides a readable performance measurement.

---

#### E. Sample Chunk Inspection

**26. `for chunk in chunks[:3]:`**

Iterates through the first three generated chunks.

**Code Breakdown:**
- `chunks[:3]` — Selects the first three chunks.
- `chunk` — Represents the current chunk record.

**Purpose in the RAG System:**
Allows the researcher to inspect sample chunk boundaries and text completeness.

---

**27. `print(f"Chunk ID : {chunk['id']}")`**

Displays the unique chunk identifier.

**Purpose:**
Supports chunk identification and traceability.

---

**28. `print(f"Page : {chunk['page']}")`**

Displays the source PDF page number.

**Purpose:**
Identifies where the chunk originated.

---

**29. `print(f"Characters : {len(chunk['text'])}")`**

Displays the actual chunk length.

**Purpose:**
Allows comparison between configured and generated chunk lengths.

---

**30. `print(chunk['text'])`**

Displays the full text of the sample chunk.

**Purpose in the RAG System:**
Allows manual inspection of whether vocabulary definitions and contextual information remain intact.

**Research Relevance:**
This helps identify incomplete definitions or unnecessary text within chunks.

---

#### F. Embedding Generation

**31. `t1 = time.perf_counter()`**

Records the starting time for embedding generation.

**Purpose:**
Enables measurement of embedding computation time.

---

**32. `vectors = encoder.encode(...)`**

Converts the generated document chunks into numerical vector embeddings.

`['passage: ' + x['text'] for x in chunks]`

- Retrieves the text of each chunk.
- Adds the `passage: ` prefix.
- Prepares the text for the E5 embedding model.

**Why `passage:` is used:**
The multilingual E5 model is designed to use different input prefixes for retrieval tasks.

- `passage:` — Document text being indexed.
- `query:` — User question being searched.

`normalize_embeddings=True`

- Normalizes the generated vectors to unit length.
- Supports consistent similarity calculations.

`batch_size=32`

- Processes up to 32 texts per encoding batch.
- Controls embedding computation batching.

`show_progress_bar=False`

- Disables the encoding progress bar.
- Keeps the notebook output concise.

**Purpose in the RAG System:**
Transforms vocabulary chunks into vectors that can be stored and compared in ChromaDB.

---

**33. `embedding_seconds = time.perf_counter() - t1`**

Calculates embedding generation time.

**Purpose:**
Measures the duration required to encode the generated document chunks.

---

#### G. ChromaDB Collection Construction

**34. `t2 = time.perf_counter()`**

Records the start of the database indexing stage.

**Purpose:**
Measures the time required to recreate and populate the vector collection.

---

**35. `try: client.delete_collection(config_id)`**

Attempts to delete an existing ChromaDB collection with the same configuration identifier.

**Purpose:**
- Prevents old records from being reused.
- Allows the collection to be rebuilt.
- Ensures that the active collection reflects the current chunking output.

---

**36. `except (ValueError, chromadb.errors.NotFoundError): pass`**

Handles the specified exceptions raised when attempting collection deletion.

**Purpose:**
Allows execution to continue when the collection does not exist or the specified
exception is encountered.

---

**37. `col = client.create_collection(...)`**

Creates a new ChromaDB collection.

**Parameters:**
- `name=config_id` — Uses the active configuration ID.
- `metadata={'hnsw:space': 'cosine'}` — Requests cosine
  distance as the vector similarity space.

**Purpose in the RAG System:**
Creates the storage structure for document embeddings and their associated information.

---

**38. `for start in range(0, len(chunks), 100):`**

Iterates through the generated chunks in groups of up to 100.

**Purpose:**
Divides database insertion into manageable batches.

---

**39. `batch = chunks[start:start + 100]`**

Selects the current batch of chunks.

**Purpose:**
Prepares a subset of chunk records for insertion into ChromaDB.

---

**40. `col.add(...)`**

Inserts the current batch into the ChromaDB collection.

The insertion includes:

`ids`

- Unique identifiers of the chunks.

`documents`

- Original text content of the chunks.

`metadatas`

- Source page number.
- Active configuration identifier.

`embeddings`

- Numerical vectors produced by the SentenceTransformer model.

**Purpose in the RAG System:**
Stores the information needed for semantic similarity retrieval.

---

**41. `vectors[start:start + 100].tolist()`**

Selects the corresponding embedding vectors and converts them into Python lists.

**Purpose:**
Provides the numerical embedding data for the current insertion batch.

---

**42. `index_seconds = time.perf_counter() - t2`**

Calculates the elapsed database indexing time.

**Purpose:**
Measures collection deletion, creation, and vector insertion time.

---

**43. `collections[config_id] = col`**

Stores the ChromaDB collection reference in the `collections` dictionary.

**Purpose in the RAG System:**
Allows later retrieval operations to access the indexed vocabulary chunks.

---

#### H. Experimental Record Management

**44. `experiment_logs.append({...})`**

Adds a summary record for the active experiment.

The record contains:

- Configuration identifier.
- Chunk size and overlap.
- Number of source pages.
- Number of generated chunks.
- Average chunk length.
- Minimum and maximum chunk lengths.
- Chunking time.
- Embedding generation time.
- Indexing time.
- Embedding vector dimension.

**Purpose in the RAG System:**
Creates a structured record of the experimental configuration and its processing performance.


---

**45. `chunk_rows.extend(...)`**

Adds individual chunk records to the `chunk_rows` list.

**Stored Information:**
- Configuration identifier.
- Chunk identifier.
- Source page number.
- Chunk text.
- Character count.

**Purpose:**
Supports chunk-level analysis and later CSV or Excel export.

---

#### I. Embedding and Database Results

**46. `print("EMBEDDING AND VECTOR DATABASE RESULTS")`**

Displays the embedding and indexing results section heading.

---

**47. `print(f"Embedding model : {EMBED_MODEL}")`**

Displays the embedding model identifier.

**Purpose:**
Confirms the model used to generate vectors.

---

**48. `print(f"Vector dimensions :
{vectors.shape[1]}")`**

Displays the dimensionality of the generated embeddings.

**Purpose:**
Verifies the vector size stored in ChromaDB.

For the selected multilingual E5-small model, the expected dimension is 384.

---

**49. `print(f"Indexed Chroma records : {col.count()}")`**

Retrieves the number of records stored
in the active ChromaDB collection.

**Purpose:**
Confirms the number of indexed document chunks.

---

**50. Embedding and Indexing Time Displays**

`print(f"Embedding seconds ...")`

`print(f"Indexing seconds ...")`

Displays the elapsed processing times.

**Purpose:**
Provides performance measurements for the embedding and indexing stages.

---

#### J. Configuration Summary

**51. `print("CURRENT CONFIGURATION SUMMARY")`**

Displays the final experiment summary heading.

---

**52. `pd.DataFrame(experiment_logs)
.to_string(index=False)`**

Converts the experiment log records
into a readable table.

**Purpose in the RAG System:**
- Summarizes the active experiment.
- Displays chunking and indexing statistics.
- Supports experiment documentation.



## 7 · Retrieval with source pages and timing


In [34]:
# ============================================================
# 7. RETRIEVAL TEST AND DETAILED RESULTS
# ============================================================
def retrieve(question, config_id=None, k=TOP_K):
    if config_id is None:
        config_id = DEFAULT_CONFIG
    if config_id not in collections:
        raise KeyError(f"Unknown configuration {config_id!r}; available: {list(collections)}")
    t0 = time.perf_counter()
    qvec = encoder.encode(['query: ' + question], normalize_embeddings=True)[0].tolist()
    result = collections[config_id].query(
        query_embeddings=[qvec], n_results=min(k, collections[config_id].count()),
        include=['documents', 'metadatas', 'distances'],
    )
    docs = [
        {'rank': i + 1, 'id': rid, 'text': txt, 'page': meta['page'],
         'distance': float(dist)}
        for i, (rid, txt, meta, dist) in enumerate(zip(
            result['ids'][0], result['documents'][0],
            result['metadatas'][0], result['distances'][0],
        ))
    ]
    return docs, time.perf_counter() - t0

q = 'Ano ang ibig sabihin ng hilahil?'
results, elapsed = retrieve(q)
print("\n" + "=" * 70)
print("TESTING CHROMA VECTOR STORE")
print("=" * 70)
print(f"Query               : {q}")
print(f"Configuration       : {DEFAULT_CONFIG}")
print(f"Retrieval seconds   : {elapsed:.4f}")
print(f"Results retrieved   : {len(results)}")
for x in results:
    print("\n" + "-" * 70)
    print(f"RESULT {x['rank']}")
    print("-" * 70)
    print(f"Chunk ID        : {x['id']}")
    print(f"Page            : {x['page']}")
    print(f"Cosine distance : {x['distance']:.4f} (lower is better)")
    print("-" * 70)
    print(x['text'])


TESTING CHROMA VECTOR STORE
Query               : Ano ang ibig sabihin ng hilahil?
Configuration       : CFG3
Retrieval seconds   : 0.0675
Results retrieved   : 3

----------------------------------------------------------------------
RESULT 1
----------------------------------------------------------------------
Chunk ID        : CFG3_p5_c2
Page            : 5
Cosine distance : 0.1132 (lower is better)
----------------------------------------------------------------------
VOCABULARY ENTRY 015
WORD: hilahil
PART_OF_SPEECH: pangngalan
DEFINITION: dusa, bagabag, o malalim na pagdurusa at ligalig
CONTEXTUAL_MEANING: mga paghihirap at alalahaning dinanas dahil sa pag-ibig
STANZA: FL-042, FL-097; KS-03 (Kay Selya)
SOURCE: KWF Diksiyonaryo ng Wikang Filipino

----------------------------------------------------------------------
RESULT 2
----------------------------------------------------------------------
Chunk ID        : CFG3_p41_c0
Page            : 41
Cosine distance : 0.1426 (lower i

### Purpose

This section implements and tests the semantic retrieval component of the LalimSalita Retrieval-Augmented Generation (RAG) system.

It defines a reusable retrieval function that converts a Filipino vocabulary question into a numerical embedding, searches the active ChromaDB collection, and returns the most semantically similar document chunks.

The section also measures retrieval execution time and displays the retrieved results with their rankings, chunk identifiers, source page numbers, cosine distances, and original text.

The retrieved information will serve as the supporting
document context for the language model in the subsequent
answer generation stage.

---

### Code Explanation

#### A. Define the Retrieval Function

**1. `def retrieve(question, config_id=None, k=TOP_K):`**

Defines a reusable Python function named `retrieve`.

**Parameters:**
- `question` — The user's vocabulary question.
- `config_id` — Identifier of the ChromaDB collection to search.
- `k` — Maximum number of chunks requested.

**Purpose in the RAG System:**
Provides a standardized method for retrieving relevant vocabulary chunks from the selected vector collection.

The default retrieval cutoff is defined by `TOP_K` in Section 4.

---

**2. `if config_id is None:`**

Checks whether a configuration identifier was provided.

**Purpose:**
Determines whether the function should use the default experimental configuration.

---

**3. `config_id = DEFAULT_CONFIG`**

Assigns the active configuration identifier when no specific configuration is provided.

**Purpose in the RAG System:**
Ensures retrieval uses the collection corresponding to the active chunking experiment.

---

**4. `if config_id not in collections:`**

Checks whether the requested configuration exists in the `collections` dictionary.

**Purpose:**
Prevents retrieval from an unavailable or uninitialized ChromaDB collection.

---

**5. `raise KeyError(...)`**

Raises an error when the requested configuration cannot be found.

**Purpose:**
- Stops the invalid retrieval request.
- Identifies the unavailable configuration.
- Displays the collection identifiers currently available in memory.

---

#### B. Query Embedding Generation

**6. `t0 = time.perf_counter()`**

Records the starting time of the retrieval operation.

**Purpose:**
Enables measurement of the combined query embedding and ChromaDB search execution time.

---

**7. `qvec = encoder.encode(['query: ' + question], normalize_embeddings=True)[0].tolist()`**

Converts the user's question into a numerical vector.

**Code Breakdown:**

`'query: ' + question`

- Adds the `query:` prefix to the question.
- Follows the input formatting convention of the multilingual E5 embedding model.

`encoder.encode(...)`

- Uses the SentenceTransformer model loaded in Section 5.
- Generates a numerical representation of the question.

`normalize_embeddings=True`

- Normalizes the query embedding to unit length.
- Supports consistent vector similarity comparison.

`[0]`

- Selects the first embedding from the returned collection of embeddings.
- Only one question is encoded in this operation.

`.tolist()`

- Converts the embedding into a Python list.
- Prepares the vector for the ChromaDB query.

**Purpose in the RAG System:**
Transforms the user's Filipino vocabulary question into a vector that can be compared with the stored document embeddings.

---

#### C. ChromaDB Similarity Search

**8. `result = collections[config_id].query(...)`**

Performs a vector similarity search against the selected ChromaDB collection.

**Purpose in the RAG System:**
Identifies stored vocabulary chunks whose embeddings are closest to the question embedding.

---

**9. `query_embeddings=[qvec]`**

Passes the generated question embedding to ChromaDB.

**Purpose:**
Uses the numerical representation of the question as the basis for similarity retrieval.

---

**10. `n_results=min(k, collections[config_id].count())`**

Determines how many results to request.

**Code Breakdown:**
- `k` — Requested retrieval cutoff.
- `.count()` — Number of records in the collection.
- `min()` — Selects the smaller value.

**Purpose:**
Prevents requesting more results than the
number of stored document chunks.

If the collection contains only two chunks, the function requests two results.

---

**11. `include=['documents', 'metadatas', 'distances']`**

Specifies the information ChromaDB should return.

**Returned Information:**
- `documents` — Retrieved vocabulary chunk text.
- `metadatas` — Associated metadata, including
  the source PDF page number.
- `distances` — Vector distances between the
  query and retrieved document embeddings.

**Purpose in the RAG System:**
Provides the content and source information needed for answer generation and manual evaluation.

---

#### D. Organize Retrieved Results

**12. `docs = [...]`**

Creates a list of structured retrieval records.

**Purpose:**
Converts ChromaDB's returned results into a consistent format for later processing.

Each record contains:

- `rank`
- `id`
- `text`
- `page`
- `distance`

---

**13. `'rank': i + 1`**

Assigns a ranking position to each retrieved chunk.

**Purpose:**
Identifies the order of retrieval results.

---

**14. `'id': rid`**

Stores the retrieved chunk identifier.

**Purpose:**
Allows the researcher to identify the exact document chunk returned by ChromaDB.

---

**15. `'text': txt`**

Stores the retrieved chunk's original text.

**Purpose in the RAG System:**
Provides the vocabulary information that may be supplied to the language model as context.

---

**16. `'page': meta['page']`**

Retrieves the source PDF page number from the stored chunk metadata.

**Purpose:**
Supports source identification and page-based references.

**Important:**
The page number represents the PDF page index assigned during extraction in Section 3.

---

**17. `'distance': float(dist)`**

Stores the returned vector distance as a floating-point number.

**Purpose:**
Provides a numerical measure of vector similarity.

Because the collection uses cosine distance, smaller values generally indicate greater semantic similarity.



---

**18. `for i, (rid, txt, meta, dist) in enumerate(zip(...))`**

Iterates through the retrieved result fields.

- `zip()` — Groups corresponding chunk IDs, texts, metadata, and distances.
- `enumerate()` — Assigns a position to each result.
- `i` — Zero-based result index.
- `rid` — Retrieved chunk identifier.
- `txt` — Retrieved chunk text.
- `meta` — Associated chunk metadata.
- `dist` — Returned vector distance.

**Purpose:**
Combines the separate ChromaDB output fields into individual structured retrieval records.

---

**19. `return docs, time.perf_counter() - t0`**

Returns the retrieval results and elapsed time.

**Returned Values:**
- `docs` — List of retrieved chunk records.
- Elapsed seconds — Time spent generating
  the query embedding and searching ChromaDB.

**Purpose in the RAG System:**
Makes the retrieval results available to other notebook sections, including RAG answer generation.

---

#### E. Execute a Test Retrieval

**20. `q = 'Ano ang ibig sabihin ng hilahil?'`**

Defines a sample Filipino vocabulary question.

**Purpose:**
Provides a test input for checking whether the retrieval pipeline is functioning.

---

**21. `results, elapsed = retrieve(q)`**

Executes the retrieval function using the test question.

- `retrieve(q)` — Searches the active collection.
- `results` — Stores the retrieved chunk records.
- `elapsed` — Stores retrieval execution time.

**Purpose:**
Tests the complete query embedding and vector retrieval process.

---

#### F. Display Retrieval Summary

**22. `print("TESTING CHROMA VECTOR STORE")`**

Displays the retrieval test heading.

**Purpose:**
Identifies the current notebook operation.

---

**23. `print(f"Query : {q}")`**

Displays the submitted vocabulary question.

**Purpose:**
Documents the input used for the retrieval test.

---

**24. `print(f"Configuration : {DEFAULT_CONFIG}")`**

Displays the active chunking configuration.

**Purpose:**
Identifies which ChromaDB collection was searched.

---

**25. `print(f"Retrieval seconds : {elapsed:.4f}")`**

Displays retrieval execution time to four decimal places.

**Purpose:**
Measures the response time of the retrieval stage.

---

**26. `print(f"Results retrieved :
{len(results)}")`**

Displays the number of returned chunks.

**Purpose:**
Confirms how many retrieval results were produced by ChromaDB.

---

#### G. Display Detailed Retrieved Chunks

**27. `for x in results:`**

Iterates through every retrieved chunk.

**Purpose:**
Displays the results individually for manual inspection.

---

**28. `print(f"RESULT {x['rank']}")`**

Displays the retrieval rank.

**Purpose:**
Makes the ordering of retrieved chunks visible.

---

**29. `print(f"Chunk ID : {x['id']}")`**

Displays the unique chunk identifier.

**Purpose:**
Supports traceability between retrieved results and stored document chunks.

---

**30. `print(f"Page : {x['page']}")`**

Displays the source PDF page number.

**Purpose:**
Allows the researcher to locate the retrieved information in the original document.

---

**31. `print(f"Cosine distance : {x['distance']:.4f} (lower is better)")`**

Displays the cosine distance between the query embedding and document embedding.

**Purpose:**
Provides a numerical indication ofsemantic similarity.

---

**32. `print(x['text'])`**

Displays the complete text of the retrieved chunk.

**Purpose in the RAG System:**
- Allows inspection of retrieved vocabulary content.
- Helps identify whether the correct definition was retrieved.
- Supports manual relevance assessment.
- Provides the source context that can later be passed to the language model.


## 8 · Secure Groq API setup

In [35]:
if not os.environ.get('GROQ_API_KEY'):
    try:
        from google.colab import userdata
        secret = userdata.get('GROQ_API_KEY')
        if secret: os.environ['GROQ_API_KEY'] = secret
    except Exception: pass
if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Enter GROQ_API_KEY (hidden): ')
groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])
print('Groq client initialized. API key not displayed or saved in notebook.')

Groq client initialized. API key not displayed or saved in notebook.


### Purpose

This section initializes the Groq API client used for language model answer generation in the LalimSalita Retrieval-Augmented Generation (RAG) system.

It securely retrieves the Groq API key from the runtime environment or Google Colab Secrets.

If the API key is unavailable from both sources, the system prompts the user to enter it manually using a hidden input field.

Once the API key is available, the Groq client is initialized and becomes ready to send requests to the selected language model.

The API key is not intentionally printed in the notebook output.

---

### Code Explanation

#### A. Check Existing Environment Variable

**1. `if not os.environ.get('GROQ_API_KEY'):`**

Checks whether the Groq API key is already available in the environment variables.

**Code Breakdown:**
- `os.environ` — Provides access to environment variables in the current Python process.
- `.get('GROQ_API_KEY')` — Retrieves the stored API key if it exists.
- `not` — Checks whether the value is missing or empty.

**Purpose in the RAG System:**
Prevents unnecessary API key requests when a key is already available.

**Important:**
If the key already exists, the code skips Google Colab Secrets and manual input.

---

#### B. Retrieve API Key from Google Colab Secrets

**2. `try:`**

Begins an exception-handling block.

**Purpose:**
Allows the system to attempt retrieving the API key from Google Colab without immediately stopping execution if retrieval fails.

---

**3. `from google.colab import userdata`**

Imports Google Colab's userdata module.

**Purpose:**
Provides access to secrets configured through Google Colab's Secrets interface.

---

**4. `secret = userdata.get('GROQ_API_KEY')`**

Attempts to retrieve the Groq API key from Google Colab Secrets.

- `userdata.get()` — Requests a stored secret.
- `'GROQ_API_KEY'` — Name of the secret.
- `secret` — Stores the retrieved value.

**Purpose in the RAG System:**
Obtains the credential needed to authenticate requests to the Groq API.

---

**5. `if secret:`**

Checks whether a nonempty secret was retrieved.

**Purpose:**
Prevents an empty secret from being assigned as the API key.

---

**6. `os.environ['GROQ_API_KEY'] = secret`**

Stores the retrieved secret in the current Python process's environment variables.

**Purpose in the RAG System:**
Makes the API key available for initializing the Groq client.

---

**7. `except Exception: pass`**

Catches exceptions raised while attempting to access Google Colab Secrets.

**Code Breakdown:**
- `except Exception` — Handles ordinary exceptions raised inside the try block.
- `pass` — Continues execution without displaying an error message.

**Purpose:**
Allows the program to use manual API key entry if Google Colab Secrets cannot
provide the credential.

---

#### C. Manual API Key Input

**8. `if not os.environ.get('GROQ_API_KEY'):`**

Checks again whether an API key is available.

**Purpose:**
Determines whether manual input is required after the environment and Colab Secrets checks.

---

**9. `getpass.getpass(
'Enter GROQ_API_KEY (hidden): ')`**

Prompts the user to enter the Groq API key without displaying the typed characters.

**Code Breakdown:**
- `getpass` — Python module for hidden input.
- `.getpass()` — Requests sensitive input without echoing the characters.
- Prompt string — Instructs the user to enter the API key.

**Purpose in the RAG System:**
Provides an alternative authentication method when no stored API key is available.

---

**10. `os.environ['GROQ_API_KEY'] = getpass.getpass(...)`**

Stores the manually entered API key in the current process environment.

**Purpose:**
Makes the credential available to the Groq client initialization process.

---

#### D. Initialize Groq Client

**11. `groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])`**

Creates a Groq API client using the available API key.

**Code Breakdown:**
- `Groq()` — Initializes the Groq Python SDK client.
- `api_key=` — Supplies the authentication credential.
- `os.environ['GROQ_API_KEY']` — Retrieves the key from the current process environment.
- `groq_client` — Stores the initialized client.

**Purpose in the RAG System:**
Prepares the system to send language model generation requests through the Groq API.

---

#### E. Display Initialization Confirmation

**12. `print('Groq client initialized. API key not displayed or saved in notebook.')`**

Displays a confirmation message after the Groq client object has been created.

**Purpose:**
- Confirms that initialization completed.
- Avoids printing the API key.
- Indicates that the key was not intentionally written into the notebook source.

## 9 · Grounded answer generation and RAG chatbot

In [36]:
SYSTEM_PROMPT = (
    "You are LalimSalita, a Filipino vocabulary assistant. "
    "Answer only from the supplied PDF excerpts. "
    "Explain the requested vocabulary word using its definition and contextual meaning when available. "
    "Do not invent definitions, interpretations, examples, sources, or page numbers. "
    "Cite the supporting PDF page numbers in the format (Pahina X). "
    "If the evidence does not support the answer, respond: "
    "'Walang sapat na impormasyon sa ibinigay na dokumento.' "
    "Answer in Filipino using clear, concise, and natural language."
)


def rag_query(question, config_id=None, k=TOP_K):
    if config_id is None:
        config_id = DEFAULT_CONFIG
    docs, retrieval_seconds = retrieve(question, config_id, k)
    context='\n\n'.join(f"[Source: page {d['page']}, rank {d['rank']}]\n{d['text']}" for d in docs)
    t0=time.perf_counter()
    completion=groq_client.chat.completions.create(
        model=GROQ_MODEL, temperature=TEMPERATURE, max_completion_tokens=500,
        messages=[{'role':'system','content':SYSTEM_PROMPT},
                  {'role':'user','content':f'DOCUMENT CONTEXT:\n{context}\n\nQUESTION: {question}'}])
    generation_seconds=time.perf_counter()-t0
    return {'question':question,'config':config_id,'answer':completion.choices[0].message.content,
            'retrieved':docs,'retrieval_seconds':retrieval_seconds,
            'generation_seconds':generation_seconds,
            'prompt_tokens':getattr(completion.usage,'prompt_tokens',None),
            'completion_tokens':getattr(completion.usage,'completion_tokens',None)}

sample = rag_query('Ano ang ibig sabihin ng hilahil?')
print('\n' + '=' * 70)
print('GROQ RAG CHATBOT TEST')
print('=' * 70)
print('Question      :', sample['question'])
print('Configuration :', sample['config'])
print('Answer        :', sample['answer'])
print('Source pages  :', [d['page'] for d in sample['retrieved']])
print('Retrieval sec :', round(sample['retrieval_seconds'], 4))
print('Generation sec:', round(sample['generation_seconds'], 4))


GROQ RAG CHATBOT TEST
Question      : Ano ang ibig sabihin ng hilahil?
Configuration : CFG3
Answer        : Ang **hilahil** ay pangngalang tumutukoy sa **dusa, bagabag, o malalim na pagdurusa at ligalig**—karaniwang mga paghihirap at alalahanin na dinanas dahil sa pag-ibig. (Pahina 5)
Source pages  : [5, 41, 57]
Retrieval sec : 0.0529
Generation sec: 0.4908


### Purpose

This section implements the complete Retrieval-Augmented
Generation (RAG) question-answering pipeline of the
LalimSalita Filipino vocabulary assistant.

It integrates the semantic retrieval function from
Section 7 with the Groq language model client
initialized in Section 8.

When a user submits a vocabulary question, the system
retrieves the most semantically similar document chunks
from ChromaDB and constructs a context containing the
retrieved text and its source-page information.

The context and question are then submitted to the
language model, which is instructed to answer in Filipino
using only the provided PDF excerpts.

This section also records retrieval time, answer
generation time, and API token usage.

---

### Code Explanation

#### A. Define the System Prompt

**1. `SYSTEM_PROMPT = """..."""`**

Defines the system-level instructions used to guide the language model's behavior.

**Purpose in the RAG System:**
- Establishes the chatbot's identity as LalimSalita.
- Specifies Filipino as the response language.
- Restricts answers to retrieved PDF excerpts.
- Requires source-page citations.
- Instructs the model not to invent definitions.
- Establishes a fallback response when evidence is insufficient.

**System Instructions:**

`You are LalimSalita, a Filipino vocabulary assistant.`

Defines the chatbot's role as an assistant specializing in Filipino vocabulary.

`Answer in Filipino using ONLY the provided PDF excerpts.`

Instructs the model to generate answers based only on the retrieved document context.

`Explain the vocabulary term and cite page numbers.`

Requires the model to explain vocabulary terms and reference their source PDF pages.

`If the excerpts do not support an answer, explicitly say: 'Walang sapat na impormasyon sa ibinigay na dokumento.'`

Defines the response to use when the retrieved context does not provide sufficient evidence.

`Do not invent vocabulary definitions or references.`

Instructs the model to avoid unsupported definitions and fabricated citations.

---

#### B. Define the RAG Query Function

**2. `def rag_query(question, config_id=None, k=TOP_K):`**

Defines a reusable function named `rag_query`.

**Parameters:**
- `question` — User's Filipino vocabulary question.
- `config_id` — ChromaDB configuration to use.
- `k` — Maximum number of chunks to retrieve.

**Purpose in the RAG System:**
Combines semantic retrieval and language model answer generation into one function.

---

**3. `if config_id is None:`**

Checks whether a specific chunking configuration was provided.

**Purpose:**
Determines whether the active default configuration should be used.

---

**4. `config_id = DEFAULT_CONFIG`**

Assigns the active configuration identifier.

**Purpose:**
Ensures the RAG function searches the collection associated with the selected experiment.

---

#### C. Retrieve Relevant Document Chunks

**5. `docs, retrieval_seconds = retrieve( question, config_id, k)`**

Calls the retrieval function defined in Section 7.

**Returned Values:**
- `docs` — Ranked document chunks retrieved
  from ChromaDB.
- `retrieval_seconds` — Time spent generating
  the query embedding and searching ChromaDB.

**Purpose in the RAG System:**
Retrieves the vocabulary information that will serve as supporting context for the language model.

---

#### D. Construct the Retrieved Context

**6. `context = '\n\n'.join(...)`**

Combines the retrieved chunks into one formatted context string.

`'\n\n'.join(...)`

Joins the retrieved chunks using two newline characters to separate the source excerpts.

`f"[Source: page {d['page']}, rank {d['rank']}]"`

Adds the source page and retrieval rank
before each chunk.

`d['text']`

Contains the original text of the retrieved chunk.

`for d in docs`

Iterates through every retrieved document chunk.

**Purpose in the RAG System:**
- Organizes retrieved information for the LLM.
- Preserves source-page information.
- Separates individual document excerpts.
- Provides the language model with evidence for generating its response.

**Important:**
The page numbers shown above are illustrative. Actual source pages depend on the retrieved results.

---

#### E. Measure Generation Time

**7. `t0 = time.perf_counter()`**

Records the starting time before the
Groq language model request.

**Purpose:**
Enables measurement of answer generation latency.

---

#### F. Generate an Answer Through Groq

**8. `completion = groq_client.chat.completions.create(...)`**

Sends the retrieved context and user question to the Groq API for answer generation.

**Purpose in the RAG System:**
Uses the selected language model to construct a Filipino answer based on the retrieved evidence.

---

**9. `model=GROQ_MODEL`**

Specifies the language model identifier defined in Section 4.

**Purpose:**
Ensures the same model is used across the chunking experiments.

---

**10. `temperature=TEMPERATURE`**

Passes the configured generation temperature.

**Purpose:**
Controls the model's sampling behavior, subject to the API and model's support.

A relatively low temperature is intended to encourage more consistent responses.

---

**11. `max_completion_tokens=500`**

Sets the maximum number of completion tokensallowed for the model response.

**Purpose:**
Limits the length of the generated completion.


---

**12. `messages=[...]`**

Defines the messages sent to the language model.

The request contains two messages:

**System Message**

`{'role': 'system', 'content': SYSTEM_PROMPT}`

Provides the chatbot's behavior and answer-grounding instructions.

**User Message**

`{'role': 'user','content': f'DOCUMENT CONTEXT:\n{context} \n\nQUESTION: {question}'}`

Supplies the retrieved document excerpts and the vocabulary question.

**Purpose in the RAG System:**
Allows the model to distinguish the answer-generation instructions from
the question and supporting context.

---

#### G. Calculate Generation Time

**13. `generation_seconds = time.perf_counter() - t0`**

Calculates the elapsed time for the Groq API completion request.

**Purpose:**
Measures the duration of the answer-generation request, including API communication latency.

---

#### H. Return Structured RAG Results

**14. `return {...}`**

Returns a dictionary containing the question, generated answer, retrieved chunks, timing information, and token usage.

**Returned Fields:**

`'question': question`

Stores the original vocabulary question.

`'config': config_id`

Stores the active chunking configuration.

`'answer': completion.choices[0].message.content`

Extracts the generated response text from the first completion choice.

`'retrieved': docs`

Stores the ranked document chunks used as supporting context.

`'retrieval_seconds': retrieval_seconds`

Stores the elapsed retrieval time.

`'generation_seconds': generation_seconds`

Stores the elapsed generation request time.

`'prompt_tokens':
getattr(completion.usage, 'prompt_tokens', None)`

Retrieves the reported number of input tokens when that information is available.

`'completion_tokens':
getattr(completion.usage, 'completion_tokens', None)`

Retrieves the reported number of output tokens when that information is available.

**Purpose in the RAG System:**
Provides a structured response that can be
displayed, logged, or exported for analysis.

---

#### I. Execute a Sample RAG Question

**15. `sample = rag_query(
'Ano ang ibig sabihin ng hilahil?')`**

Executes the complete RAG pipeline
using a sample Filipino vocabulary question.

**Purpose:**
Tests the integration of:
- Query embedding.
- ChromaDB retrieval.
- Context construction.
- Groq answer generation.

---

#### J. Display the RAG Test Results

**16. `print('GROQ RAG CHATBOT TEST')`**

Displays the test section heading.

---

**17. `print('Question :', sample['question'])`**

Displays the original vocabulary question.

**Purpose:**
Identifies the question submitted to the system.

---

**18. `print('Configuration :', sample['config'])`**

Displays the active chunking configuration.

**Purpose:**
Identifies which vector collection was used to retrieve the evidence.

---

**19. `print('Answer :', sample['answer'])`**

Displays the language model's generated answer.

**Purpose:**
Allows the researcher to inspect whether the answer explains the vocabulary term
and follows the grounding instructions.

---

**20. `print('Source pages :', [d['page'] for d in sample['retrieved']])`**

Displays the source page numbers of all retrieved document chunks.

**Purpose:**
Supports source tracing and manual verification.

---

**21. `print('Retrieval sec :', round(sample['retrieval_seconds'], 4))`**

Displays the retrieval time rounded to four decimal places.

**Purpose:**
Measures the latency of the semantic retrieval stage.

---

**22. `print('Generation sec:',
round(sample['generation_seconds'], 4))`**

Displays the generation request time rounded to four decimal places.

**Purpose:**
Measures the latency of the language model answer-generation stage.

## 9A · Continuous question-and-answer session (interactive)

In [37]:
''

# ============================================================
# 9A. CONTINUOUS RAG QUESTION-ANSWER LOOP (CHROMADB + GROQ)
# ============================================================
# Run this cell on its own after the Groq client and collections
# have been initialized. Type 'exit' to end the input loop.

active_config = DEFAULT_CONFIG
qa_history = []
retrieval_history = []

print("\n" + "=" * 70)
print("CONTINUOUS RAG QUESTION ANSWERING — LALIMSALITA")
print("=" * 70)
print("Ask Filipino vocabulary questions about the uploaded PDF.")
print("Command: exit")
print(f"Active chunking configuration: {active_config}")

while True:
    try:
        question = input("\nQuestion: ").strip()

    except (EOFError, KeyboardInterrupt):
        print("\nRAG session terminated.")
        break

    if not question:
        print("Please enter a question.")
        continue

    # --------------------------------------------------------
    # EXIT COMMAND
    # --------------------------------------------------------
    if question.lower() == "exit":
        print("\nRAG session terminated.")
        print(f"Total questions answered: {len(qa_history)}")
        break

    # --------------------------------------------------------
    # PROCESS RAG QUESTION
    # --------------------------------------------------------
    try:
        response = rag_query(
            question,
            config_id=active_config,
            k=TOP_K
        )

        # ----------------------------------------------------
        # SAVE QUESTION-ANSWER HISTORY
        # ----------------------------------------------------
        qa_history.append({
            'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'config': active_config,
            'question': question,
            'answer': response['answer'],
            'retrieval_seconds': response['retrieval_seconds'],
            'generation_seconds': response['generation_seconds'],
            'prompt_tokens': response['prompt_tokens'],
            'completion_tokens': response['completion_tokens'],
        })

        # ----------------------------------------------------
        # SAVE RETRIEVAL HISTORY
        # ----------------------------------------------------
        for doc in response['retrieved']:
            retrieval_history.append({
                'config': active_config,
                'question': question,
                'rank': doc['rank'],
                'page': doc['page'],
                'chunk_id': doc['id'],
                'cosine_distance': doc['distance'],
                'retrieved_text': doc['text'],
            })

        # ----------------------------------------------------
        # DISPLAY RAG ANSWER
        # ----------------------------------------------------
        print("\n" + "=" * 70)
        print("RAG ANSWER")
        print("=" * 70)

        print(response['answer'])

        print(
            f"\nConfiguration: {response['config']} | "
            f"Retrieval: {response['retrieval_seconds']:.4f}s | "
            f"Generation: {response['generation_seconds']:.4f}s"
        )

        # ----------------------------------------------------
        # DISPLAY RETRIEVED SOURCES
        # ----------------------------------------------------
        print("\n" + "=" * 70)
        print("RETRIEVED SOURCES")
        print("=" * 70)

        for doc in response['retrieved']:
            print(
                f"\n[{doc['rank']}] "
                f"Page {doc['page']} | "
                f"Chunk {doc['id']} | "
                f"Cosine distance {doc['distance']:.4f}"
            )

            print("-" * 70)
            print(doc['text'])

        print("\nType 'exit' to end the session.")

    # --------------------------------------------------------
    # ERROR HANDLING
    # --------------------------------------------------------
    except Exception as exc:
        print(f"\nERROR: {type(exc).__name__}: {exc}")
        print(
            "The question could not be processed. "
            "Check your API key, rate limits, and collections."
        )



CONTINUOUS RAG QUESTION ANSWERING — LALIMSALITA
Ask Filipino vocabulary questions about the uploaded PDF.
Command: exit
Active chunking configuration: CFG3

Question: Ano ang ibig sabihin ng salitang hilahil?

RAG ANSWER
Ang salitang **hilahil** ay tumutukoy sa “dusa, bagabag, o malalim na pagdurusa at ligalig” (Pahina 5). Sa kontekstong pangkasaysayan, ito ay naglalarawan ng mga paghihirap at alalahaning dinanas dahil sa pag-ibig (Pahina 5).

Configuration: CFG3 | Retrieval: 0.0616s | Generation: 0.5549s

RETRIEVED SOURCES

[1] Page 5 | Chunk CFG3_p5_c2 | Cosine distance 0.1091
----------------------------------------------------------------------
VOCABULARY ENTRY 015
WORD: hilahil
PART_OF_SPEECH: pangngalan
DEFINITION: dusa, bagabag, o malalim na pagdurusa at ligalig
CONTEXTUAL_MEANING: mga paghihirap at alalahaning dinanas dahil sa pag-ibig
STANZA: FL-042, FL-097; KS-03 (Kay Selya)
SOURCE: KWF Diksiyonaryo ng Wikang Filipino

[2] Page 41 | Chunk CFG3_p41_c0 | Cosine distance 0.1419

### Purpose

This section implements an interactive, continuous question-answering interface for the LalimSalita Retrieval-Augmented Generation (RAG) system.

It allows users to submit multiple Filipino vocabulary questions without repeatedly executing the RAG function.

For every valid question, the system performs semantic retrieval through ChromaDB, generates a Filipino answer through the Groq language model, and automatically displays the retrieved document sources.

The system also records question-answer interactions, retrieved chunk information, execution times, and token usage for later analysis and evaluation.

The interaction continues until the user entersthe `exit` command or interrupts the session.

---

### Code Explanation

#### A. Initialize the Interactive Session

**1. `active_config = DEFAULT_CONFIG`**

Assigns the default chunking configuration
to the interactive RAG session.

**Purpose in the RAG System:**
- Selects the ChromaDB collection to search.
- Maintains a consistent configuration throughout the question-answering session.
- Associates recorded interactions with the active chunking experiment.

---

**2. `qa_history = []`**

Creates an empty list for storing question-answer interaction records.

**Purpose:**
Records successful RAG interactions, including questions, generated answers, execution times, and token usage.

---

**3. `retrieval_history = []`**

Creates an empty list for storing retrieved document chunk records.

**Purpose:**
Preserves information about the chunks returned for each successfully processed question.

---

#### B. Display Session Information

**4. `print("\n" + "=" * 70)`**

Displays a blank line and horizontal separator.

**Purpose:**
Improves readability of the notebook output.

---

**5. `print("CONTINUOUS RAG QUESTION ANSWERING
— LALIMSALITA")`**

Displays the interactive session heading.

**Purpose:**
Identifies the LalimSalita chatbot interface.

---

**6. `print("Ask Filipino vocabulary questions about the uploaded PDF.")`**

Displays instructions describing the supported question domain.

**Purpose:**
Informs users that questions should relate to the vocabulary content of the uploaded PDF.

---

**7. `print("Command: exit")`**

Displays the available session command.

**Purpose:**
Informs users how to terminate the interactive question-answering loop.

---

**8. `print(f"Active chunking configuration:{active_config}")`**

Displays the active experimental configuration.

**Purpose:**
Confirms which ChromaDB collection will be used during the session.

---

#### C. Start the Continuous Input Loop

**9. `while True:`**

Creates an indefinite loop that repeatedly accepts user questions.

**Purpose in the RAG System:**
Allows multiple questions to be processed within one notebook execution.

The loop continues until a termination condition is encountered.

---

**10. `try:`**

Begins an exception-handling block for receiving user input.

**Purpose:**
Allows the program to handle interruptions without producing an unhandled input error.

---

**11. `question = input("\nQuestion: ").strip()`**

Displays an input prompt and reads the user's vocabulary question.

- `input()` — Accepts typed user input.
- `"\nQuestion: "` — Displays the question prompt.
- `.strip()` — Removes leading and trailing whitespace.
- `question` — Stores the processed input.

**Purpose in the RAG System:**
Collects the vocabulary question that will be submitted to the RAG pipeline.

---

**12. `except (EOFError, KeyboardInterrupt):`**

Handles input-related session interruptions.

**Code Breakdown:**
- `EOFError` — Occurs when input ends unexpectedly.
- `KeyboardInterrupt` — Occurs when execution is interrupted by the user.

**Purpose:**
Allows the session to terminate gracefully during input collection.

---

**13. `print("\nRAG session terminated.")`**

Displays a termination message.

**Purpose:**
Confirms that the interactive session has ended.

---

**14. `break`**

Exits the current `while` loop.

**Purpose:**
Stops further question prompts.

---

#### D. Validate User Input

**15. `if not question:`**

Checks whether the submitted question
is empty after whitespace removal.

**Purpose:**
Prevents empty input from being submitted to the retrieval and generation pipeline.

---

**16. `print("Please enter a question.")`**

Displays a message when the input is empty.

**Purpose:**
Instructs the user to enter a valid question.

---

**17. `continue`**

Skips the remaining statements in the current loop iteration.

**Purpose:**
Returns directly to the next question prompt.

---

#### E. Exit Command

**18. `if question.lower() == "exit":`**

Checks whether the user entered the session termination command.

- `.lower()` — Converts the input to lowercase.
- `== "exit"` — Checks for the exit command.

**Purpose:**
Allows the user to end the session using `exit`, `EXIT`, or other capitalization variants.

---

**19. `print("\nRAG session terminated.")`**

Displays the session termination message.

---

**20. `print(f"Total questions answered:{len(qa_history)}")`**

Displays the number of successfully recorded question-answer interactions.

**Code Breakdown:**
- `len(qa_history)` — Counts the stored question-answer records.

**Purpose:**
Summarizes how many questions were successfully processed during the session.

**Important:**
Failed requests are not added to `qa_history` by this code.

---

**21. `break`**

Terminates the continuous question-answering loop.

---

#### F. Process the RAG Question

**22. `try:`**

Begins the exception-handling block for the RAG processing operations.

**Purpose:**
Prevents an individual processing error from automatically ending the entire session.

---

**23. `response = rag_query(question, config_id=active_config, k=TOP_K)`**

Calls the complete RAG function defined in Section 9.

- `question` — Submitted vocabulary question.
- `config_id=active_config` — Active ChromaDB collection.
- `k=TOP_K` — Number of chunks requested.
- `response` — Stores the structured RAG output.

**Purpose in the RAG System:**
Performs the complete question-answering pipeline:

1. Generate the question embedding.
2. Search the ChromaDB vector collection.
3. Retrieve the Top-K matching chunks.
4. Construct the document context.
5. Send the context and question to Groq.
6. Generate a Filipino answer.
7. Return results and performance measurements.

---

#### G. Save Question-Answer History

**24. `qa_history.append({...})`**

Adds the successful RAG interaction to the question-answer history list.

**Purpose:**
Creates a structured record of each successfully processed question.

---

**25. `'timestamp_utc':
datetime.now(timezone.utc).isoformat()`**

Records the current date and time
using Coordinated Universal Time (UTC).

**Code Breakdown:**
- `datetime.now(timezone.utc)` — Gets the current UTC date and time.
- `.isoformat()` — Converts the timestamp into ISO 8601 format.

**Purpose:**
Provides a standardized timestamp for each question-answer record.

---

**26. `'config': active_config`**

Stores the chunking configuration used for the interaction.

**Purpose:**
Associates the answer with its experimental setup.

---

**27. `'question': question`**

Stores the user's submitted question.

**Purpose:**
Preserves the original input for evaluation.

---

**28. `'answer': response['answer']`**

Stores the generated Filipino answer.

**Purpose:**
Allows later inspection of answer quality, accuracy, and document grounding.

---

**29. `'retrieval_seconds': response['retrieval_seconds']`**

Stores the measured retrieval latency.

**Purpose:**
Records the time required for query embedding and ChromaDB similarity search.

---

**30. `'generation_seconds': response['generation_seconds']`**

Stores the measured Groq generation latency.

**Purpose:**
Records the time required for the language model completion request.

---

**31. `'prompt_tokens':
response['prompt_tokens']`**

Stores the reported input token usage.

**Purpose:**
Supports analysis of the number of tokens submitted to the language model.

---

**32. `'completion_tokens': response['completion_tokens']`**

Stores the reported output token usage.

**Purpose:**
Supports analysis of generated token consumption.

---

#### H. Save Retrieval History

**33. `for doc in response['retrieved']:`**

Iterates through every retrieved document chunk.

**Purpose:**
Processes the source records associated with the current question.

---

**34. `retrieval_history.append({...})`**

Adds each retrieved chunk to the retrieval history list.

**Purpose:**
Preserves the individual retrieval results for later inspection and evaluation.

---

**35. `'config': active_config`**

Stores the configuration used for retrieval.

---

**36. `'question': question`**

Associates the retrieved chunk with the corresponding vocabulary question.

---

**37. `'rank': doc['rank']`**

Stores the chunk's retrieval ranking.

**Purpose:**
Provides the ranking information needed for rank-based retrieval evaluation.

---

**38. `'page': doc['page']`**

Stores the original source PDF page number.

**Purpose:**
Supports source tracing and verification.

---

**39. `'chunk_id': doc['id']`**

Stores the unique retrieved chunk identifier.

**Purpose:**
Allows the exact retrieved chunk to be identified.

---

**40. `'cosine_distance': doc['distance']`**

Stores the cosine distance returned by ChromaDB.

**Purpose:**
Preserves the vector distance measurement associated with the retrieved chunk.

---

**41. `'retrieved_text': doc['text']`**

Stores the full retrieved document chunk.

**Purpose:**
Allows manual relevance judgment without having to rerun the same retrieval request.

---

#### I. Display the Generated RAG Answer

**42. `print("RAG ANSWER")`**

Displays the answer section heading.

**Purpose:**
Separates the generated response from the retrieval information.

---

**43. `print(response['answer'])`**

Displays the Filipino answer generated by the Groq language model.

**Purpose:**
Presents the answer to the user's question.

**Important:**
The answer is generated using the retrieved document excerpts supplied in Section 9.

---

**44. `print(f"\nConfiguration: ...")`**

Displays the configuration identifier, retrieval time, and generation time.

**Purpose:**
Provides immediate information about the experimental setup and execution latency.

---

#### J. Automatically Display Retrieved Sources

**45. `print("RETRIEVED SOURCES")`**

Displays the retrieved source section heading.

**Purpose:**
Separates supporting document excerpts from the generated answer.

---

**46. `for doc in response['retrieved']:`**

Iterates through the retrieved chunks.

**Purpose:**
Automatically displays all returned source records after each answer.

---

**47. `print(f"\n[{doc['rank']}] ...")`**

Displays the retrieval rank, source page, chunk identifier, and cosine distance.

**Purpose:**
Provides source information that helps the researcher inspect the retrieved evidence.

**Example Structure:**

[1] Page 13 | Chunk CFG1_p13_c0 |
Cosine distance 0.1234

**Important:**
The example values are illustrative only.

---

**48. `print("-" * 70)`**

Displays a separator before the chunk text.

**Purpose:**
Improves readability of individual results.

---

**49. `print(doc['text'])`**

Displays the full text of the retrieved chunk.

**Purpose in the RAG System:**
- Shows the evidence supplied to the LLM.
- Supports manual verification of definitions.
- Helps identify incomplete or irrelevant chunks.
- Provides information for retrieval evaluation.

---

**50. `print("\nType 'exit' to end the session.")`**

Reminds the user how to terminate the interactive session.

**Purpose:**
Provides clear navigation instructions after each successful question.

---

#### K. Error Handling

**51. `except Exception as exc:`**

Catches ordinary exceptions raised during the RAG processing block.

**Purpose:**
Allows the session to continue when an individual request fails.

---

**52. `print(f"\nERROR:{type(exc).__name__}: {exc}")`**

Displays the exception type and error message.

- `type(exc).__name__` — Gets the exception class name.
- `exc` — Contains the exception details.

**Purpose:**
Provides diagnostic information for troubleshooting failed requests.

---

**53. `print("The question could not
be processed. ...")`**

Displays a troubleshooting message.

**Purpose:**
Advises the user to check possible API key, rate limit, or collection issues.



## 10 · Export audit-ready outputs

In [38]:
# ============================================================
# 10. EXPORT ACTUAL RESULTS FOR THIS CONFIGURATION
# ============================================================
# Export after ending the continuous Q&A cell with 'exit'.
# Separate run folder prevents overwriting previous experiment files.
run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RUN_DIR = ROOT / f"{DEFAULT_CONFIG}_{run_stamp}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

chunks_df = pd.DataFrame(chunk_rows)
logs_df = pd.DataFrame(experiment_logs)
responses_df = pd.DataFrame(globals().get('qa_history', []))
retrieval_df = pd.DataFrame(globals().get('retrieval_history', []))

chunks_df.to_csv(RUN_DIR / 'chunks.csv', index=False, encoding='utf-8-sig')
logs_df.to_csv(RUN_DIR / 'experiment_logs.csv', index=False)
if not responses_df.empty:
    responses_df.to_csv(RUN_DIR / 'chatbot_responses.csv', index=False, encoding='utf-8-sig')
if not retrieval_df.empty:
    retrieval_df.to_csv(RUN_DIR / 'retrieval_results.csv', index=False, encoding='utf-8-sig')

with pd.ExcelWriter(RUN_DIR / 'OMG_Project_Experiment.xlsx', engine='openpyxl') as writer:
    logs_df.to_excel(writer, sheet_name='Chunking_Results', index=False)
    chunks_df.to_excel(writer, sheet_name='Chunks', index=False)
    if not responses_df.empty:
        responses_df.to_excel(writer, sheet_name='Actual_Responses', index=False)
    if not retrieval_df.empty:
        retrieval_df.to_excel(writer, sheet_name='Retrieval_Log', index=False)

metadata = {
    'pdf_name': PDF_PATH.name,
    'pdf_sha256': hashlib.sha256(PDF_PATH.read_bytes()).hexdigest(),
    'embedding_model': EMBED_MODEL, 'groq_model': GROQ_MODEL,
    'temperature': TEMPERATURE, 'top_k': TOP_K,
    'active_configuration': ACTIVE_CONFIG, 'seed': SEED,
    'python': sys.version, 'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'continuous_qa_questions_recorded': len(responses_df),
    'retrieval_metrics_computed_in_notebook': False,
}
(RUN_DIR / 'run_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved current run in:', RUN_DIR.resolve())
for file in sorted(RUN_DIR.iterdir()):
    print('-', file.name)


Saved current run in: /content/OMG_Project_outputs/CFG3_20261109_093635_873975
- OMG_Project_Experiment.xlsx
- chatbot_responses.csv
- chunks.csv
- experiment_logs.csv
- retrieval_results.csv
- run_metadata.json


### Purpose

This section exports the experimental results generated by the LalimSalita Retrieval-Augmented Generation (RAG) system for the currently active chunking configuration.

It organizes the collected data into a uniquely named experiment folder and saves the results in CSV, Excel, and JSON formats.

The exported records include:

1. Generated document chunks and their metadata.
2. Chunking configuration and processing statistics.
3. Recorded chatbot questions and generated answers.
4. Retrieved document chunks and their rankings.
5. Experimental settings and source document information.

These files provide the data needed for manual retrieval evaluation, answer-quality assessment, and comparison of different chunking configurations.

This section should be executed after the continuous question-answering session in Section 9A has ended.

---

### Code Explanation

#### A. Generate a Unique Experiment Folder

**1. `run_stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')`**

Generates a timestamp identifying the current experimental run.

**Code Breakdown:**

- `datetime.now(timezone.utc)` — Retrieves the current date and time in UTC.
- `.strftime()` — Formats the date and time into a filename-friendly string.
- `%Y` — Four-digit year.
- `%m` — Two-digit month.
- `%d` — Two-digit day.
- `%H` — Hour in 24-hour format.
- `%M` — Minute.
- `%S` — Second.
- `%f` — Microseconds.

**Purpose:**
Creates a timestamp that helps distinguish the current run from previous experiments.

---

**2. `RUN_DIR = ROOT / f"{DEFAULT_CONFIG}_{run_stamp}"`**

Constructs the output directory path for the current experiment.

- `ROOT` — Main output folder created in Section 2A.
- `DEFAULT_CONFIG` — Active chunking configuration.
- `run_stamp` — Timestamp identifying the run.
- `/` — Path joining operator from pathlib.

**Purpose in the RAG System:**
Organizes the exported files by experimental configuration and execution time.

---

**3. `RUN_DIR.mkdir(parents=True, exist_ok=False)`**

Creates the experiment output directory.

**Parameters:**

`parents=True`

Allows missing parent directories to be created automatically.

`exist_ok=False`

Raises an error if the target directory already exists.

**Purpose:**
Reduces the risk of accidentally overwriting files from a previous experimental run.

---

#### B. Convert Experimental Records to DataFrames

**4. `chunks_df = pd.DataFrame(chunk_rows)`**

Converts the chunk records from Section 6 into a pandas DataFrame.

**Stored Information:**

- Configuration identifier.
- Chunk identifier.
- Source PDF page.
- Chunk text.
- Character count.

**Purpose:**
Prepares the generated chunk records for CSV and Excel export.

---

**5. `logs_df = pd.DataFrame(experiment_logs)`**

Converts the experimental statistics from Section 6 into a pandas DataFrame.

**Stored Information:**

- Configuration identifier.
- Chunk size.
- Chunk overlap.
- Number of source pages.
- Number of generated chunks.
- Average chunk length.
- Minimum and maximum chunk lengths.
- Chunking execution time.
- Embedding execution time.
- Indexing execution time.
- Vector dimensions.

**Purpose:**
Preserves the processing characteristics of the active chunking configuration.

---

**6. `responses_df = pd.DataFrame(globals().get('qa_history', []))`**

Converts the recorded question-answer history into a pandas DataFrame.

`globals()`

Accesses the current global namespace.

`.get('qa_history', [])`

Retrieves the `qa_history` variable if it exists.

If it does not exist, an empty list is returned.

`pd.DataFrame()`

Converts the retrieved records into a tabular structure.

**Purpose:**
Prepares the chatbot interactions for later analysis and export.

**Important:**
If Section 9A was not executed, the resulting DataFrame may be empty.

---

**7. `retrieval_df = pd.DataFrame(globals().get('retrieval_history', []))`**

Converts the recorded retrieval history into a pandas DataFrame.

**Stored Information:**

- Configuration identifier.
- Vocabulary question.
- Retrieval rank.
- Source PDF page.
- Chunk identifier.
- Cosine distance.
- Retrieved chunk text.

**Purpose:**
Preserves the evidence returned by the vector retrieval system.

---

#### C. Export CSV Files

**8. `chunks_df.to_csv(RUN_DIR / 'chunks.csv', index=False, encoding='utf-8-sig')`**

Exports the generated document chunks to a CSV file.

`to_csv()`

Writes a DataFrame into CSV format.

`RUN_DIR / 'chunks.csv'`

Specifies the destination file.

`index=False`

Prevents the pandas row index from being included as an extra column.

`encoding='utf-8-sig'`

Uses UTF-8 encoding with a byte order mark, which can improve compatibility with Excel when opening Filipino text.

**Purpose:**
Creates a reusable record of the generated vocabulary chunks.

---

**9. `logs_df.to_csv(RUN_DIR / 'experiment_logs.csv', index=False)`**

Exports the experimental processing statistics to a CSV file.

**Purpose:**
Preserves the configuration settings, chunking statistics, and execution times.

---

**10. `if not responses_df.empty:`**

Checks whether recorded chatbot responses exist.

**Purpose:**
Prevents creation of a chatbot response CSV when the DataFrame contains no records.

---

**11. `responses_df.to_csv(RUN_DIR / 'chatbot_responses.csv', index=False, encoding='utf-8-sig')`**

Exports the question-answer interaction history.

**Purpose:**
Preserves generated answers and associated performance measurements.

---

**12. `if not retrieval_df.empty:`**

Checks whether retrieval records exist.

**Purpose:**
Prevents creation of the retrieval results CSV when no retrieval history was recorded.

---

**13. `retrieval_df.to_csv(RUN_DIR / 'retrieval_results.csv', index=False, encoding='utf-8-sig')`**

Exports the retrieval history to a CSV file.

**Purpose:**
Preserves retrieved chunks and their ranking information.

---

#### D. Export Excel Workbook

**14. `with pd.ExcelWriter(RUN_DIR / 'OMG_Project_Experiment.xlsx', engine='openpyxl') as writer:`**

Creates an Excel workbook using pandas and the openpyxl engine.

`pd.ExcelWriter()`

Creates a writer for an Excel workbook.

`RUN_DIR / 'OMG_Project_Experiment.xlsx'`

Specifies the Excel output filename.

`engine='openpyxl'`

Uses openpyxl to write the Excel file.

`with`

Automatically closes the writer when the block finishes.

**Purpose:**
Combines the experimental records into one workbook with multiple worksheets.

---

**15. `logs_df.to_excel(writer, sheet_name='Chunking_Results',
index=False)`**

Creates the `Chunking_Results` worksheet.

**Purpose:**
Stores the experimental configuration and processing statistics.

---

**16. `chunks_df.to_excel(writer, sheet_name='Chunks', index=False)`**

Creates the `Chunks` worksheet.

**Purpose:**
Stores the generated vocabulary chunks and their associated metadata.

---

**17. `if not responses_df.empty:`**

Checks whether question-answer records are available.

**Purpose:**
Creates the response worksheet only when recorded interactions exist.

---

**18. `responses_df.to_excel(writer, sheet_name='Actual_Responses',
index=False)`**

Creates the `Actual_Responses` worksheet.

**Purpose:**
Stores the submitted questions, generated answers, and performance data.

---

**19. `if not retrieval_df.empty:`**

Checks whether retrieval records exist.

**Purpose:**
Creates the retrieval worksheet only when retrieval data is available.

---

**20. `retrieval_df.to_excel(writer, sheet_name='Retrieval_Log', index=False)`**

Creates the `Retrieval_Log` worksheet.

**Purpose:**
Stores the retrieved chunk rankings, source information, cosine distances, and text.

---

#### E. Create Experiment Metadata

**21. `metadata = {...}`**

Creates a dictionary containing information about the current experimental run.

**Purpose:**
Documents the settings and source information needed to interpret the exported results.

---

**22. `'pdf_name': PDF_PATH.name`**

Stores the filename of the uploaded PDF.

**Purpose:**
Identifies the source document used in the experiment.

---

**23. `'pdf_sha256':hashlib.sha256(PDF_PATH.read_bytes()).hexdigest()`**

Computes the SHA-256 hash of the source PDF.

`PDF_PATH.read_bytes()`

Reads the PDF as binary data.

`hashlib.sha256()`

Computes its SHA-256 digest.

`.hexdigest()`

Converts the digest into a hexadecimal string.

**Purpose:**
Provides a document fingerprint that can be used to compare PDF file contents between experimental runs.

---

**24. `'embedding_model': EMBED_MODEL`**

Stores the selected embedding model.

**Purpose:**
Documents the model used to generate query and passage embeddings.

---

**25. `'groq_model': GROQ_MODEL`**

Stores the selected Groq language model.

**Purpose:**
Documents the model used for answer generation.

---

**26. `'temperature': TEMPERATURE`**

Stores the configured generation temperature.

**Purpose:**
Documents a language model generation parameter.

---

**27. `'top_k': TOP_K`**

Stores the retrieval cutoff.

**Purpose:**
Identifies the maximum number of chunks requested during retrieval.

---

**28. `'active_configuration': ACTIVE_CONFIG`**

Stores the complete active chunking configuration.

**Purpose:**
Preserves the configuration identifier, chunk size, and chunk overlap.

---

**29. `'seed': SEED`**

Stores the configured random seed.

**Purpose:**
Documents the seed value used in the notebook.

**Important:**
Recording a seed does not guarantee complete reproducibility of API responses
or all model operations.

---

**30. `'python': sys.version`**

Stores the Python interpreter version.

**Purpose:**
Documents part of the software environment used to execute the experiment.

---

**31. `'timestamp_utc':datetime.now(timezone.utc).isoformat()`**

Stores the UTC timestamp at the time the metadata dictionary is created.

**Purpose:**
Documents when the experiment results were exported.

---

**32. `'continuous_qa_questions_recorded':len(responses_df)`**

Stores the number of question-answer records exported from the interactive session.

**Purpose:**
Documents the number of recorded successful interactions.

---

**33. `'retrieval_metrics_computed_in_notebook':False`**

Indicates that retrieval evaluation metrics have not been computed in the notebook.

**Purpose:**
Distinguishes raw experimental results from completed retrieval evaluation.

---

#### F. Save Metadata as JSON

**34. `(RUN_DIR / 'run_metadata.json').write_text(...)`**

Writes the experiment metadata to a JSON file.

**Purpose:**
Preserves experiment settings in a structured, machine-readable format.

---

**35. `json.dumps(metadata, ensure_ascii=False, indent=2)`**

Converts the metadata dictionary into formatted JSON text.


`json.dumps()`

Serializes the dictionary into JSON.

`ensure_ascii=False`

Preserves Unicode characters directly instead of unnecessarily escaping them.

`indent=2`

Formats the JSON using two spaces per indentation level.

**Purpose:**
Produces a readable metadata file.

---

**36. `encoding='utf-8'`**

Specifies UTF-8 encoding for the metadata file.

**Purpose:**
Supports Unicode characters in the exported JSON.

---

#### G. Display Exported Files

**37. `print('Saved current run in:', RUN_DIR.resolve())`**

Displays the absolute path of the experiment output directory.

**Purpose:**
Allows the researcher to locate the exported files.

---

**38. `for file in sorted(RUN_DIR.iterdir()):`**

Iterates through the files inside the experiment folder.

`RUN_DIR.iterdir()`

Retrieves the directory contents.

`sorted()`

Sorts the entries.

`for file in ...`

Iterates through the entries.

**Purpose:**
Lists the generated export files in a consistent order.

---

**39. `print('-', file.name)`**

Displays the filename of each exported item.

**Purpose:**
Confirms which files were created during the export process.


## 11 · Optional ZIP download in Colab

In [40]:
import shutil
if 'RUN_DIR' not in globals():
    raise RuntimeError('Run the export cell before creating the ZIP.')
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=str(RUN_DIR))
print('Archive:', archive)
# In Colab, uncomment these lines to download:
# from google.colab import files
# files.download(archive)

Archive: /content/OMG_Project_outputs/CFG3_20261109_093635_873975.zip


### Purpose

This section creates a compressed ZIP archive containing the exported experimental results of the active LalimSalita RAG configuration.

It uses Python's built-in `shutil` module to package the CSV, Excel, and JSON files generated in Section 10 into a single archive.

Before creating the archive, the code verifies that the `RUN_DIR` variable exists in the current Python environment.

The ZIP archive can then be downloaded from Google Colab for backup, documentation, and subsequent experimental evaluation.

---

### Code Explanation

#### A. Import the Archive Utility

**1. `import shutil`**

Imports Python's built-in `shutil` module.

**Purpose:**
Provides utilities for file operations, including creating compressed archives.

**Role in the RAG System:**
Supports packaging and preservation of experimental outputs rather than retrieval or answer generation.

---

#### B. Verify the Export Directory Variable

**2. `if 'RUN_DIR' not in globals():`**

Checks whether the variable `RUN_DIR` exists in the current global Python namespace.

- `globals()` — Returns the current global namespace as a dictionary.
- `'RUN_DIR'` — Name of the variable containing the experiment output directory.
- `not in` — Checks whether that variable is absent from the namespace.

**Purpose:**
Prevents archive creation when the export directory variable has not been initialized.

---

**3. `raise RuntimeError(
'Run the export cell before creating the ZIP.')`**

Raises an exception if `RUN_DIR` is unavailable.

**Purpose:**
Stops execution and informs the researcher that Section 10 must be executed first.



---

#### C. Create the ZIP Archive

**4. `archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=str(RUN_DIR))`**

Creates a ZIP archive containing the files inside the current experiment directory.

`shutil.make_archive()`

Creates an archive from a specified directory.

`str(RUN_DIR)`

Provides the base filename for the archive.

`'zip'`

Specifies ZIP as the archive format.

`root_dir=str(RUN_DIR)`

Specifies the directory whose contents will be included in the archive.

`archive`

Stores the path of the created ZIP file.

**Purpose:**
Packages the experimental results into a single compressed file.

---

#### D. Display Archive Location

**5. `print('Archive:', archive)`**

Displays the generated ZIP archive path.

**Purpose:**
Confirms the location of the created archive.

**Expected Output:**

`Archive: OMG_Project_outputs/CFG1_20261109_071500_123456.zip`

The actual path depends on the experiment folder created in Section 10.

---

#### E. Optional Google Colab Download

**6. `# from google.colab import files`**

Contains a commented import statement for Google Colab's file utility.

**Purpose:**
Provides an optional method for downloading the archive to the user's computer.

Because the line is commented, it is not executed automatically.

---

**7. `# files.download(archive)`**

Contains a commented download command.

**Purpose:**
Allows the generated ZIP archive to be downloaded through the Google Colab interface.

To enable downloading, remove the `#` characters from both download-related lines.